# CIFAR-10 Input-Distribution Drift Benchmark — train/validation/test study (20 seeds)

Implements the finalized per-seed pipeline: **shared frozen feature extractors
with no CIFAR-10 training anywhere** — the primary is an off-the-shelf
**ImageNet-pretrained ResNet-18 (frozen)**. Support code for an untrained
random-init control is present but disabled in this 20-seed comparison. Per seed,
training-only PCA bases and references, per-(block, method) thresholds selected
on validation identities by empirical event-interval search to **AIET = 370
50-sample-window equivalents** (±10%),
a per-seed **VAE reconstruction-error detector** calibrated by the same search,
and 16 canonical test streams (12 image-transformation streams = four temporal
patterns × three transforms; four OOC-contamination streams use the same
patterns on the OOC-injection rate) plus
**severity-scaled variants of the four sudden setups** (`severity_levels`,
default s ∈ {0.25, 0.5}; the canonical streams are the s = 1 level) for the
§5c drift-magnitude response. The raw pipeline evaluates twelve projection/chart
methods per feature block plus the pixel-space VAE detector. The manuscript
retains eleven methods in total and excludes the two legacy CUSUM variants
because their realised held-out false-alarm-event rates were not adequately
matched.

**Detector geometry (fixed design decision).** Every method compares a
**fixed IC reference sample** (n = `ref_n`, drawn per seed from the same
epoch-cycled IC scheme via a dedicated RNG continuation) against **test windows
of 50 samples**, producing one statistic per window; an alarm is the statistic
crossing its calibrated threshold. Window starts are spaced `Config.stride`
samples apart — **default stride = 1, i.e. maximally sliding windows**: every
sample starts a new window, consecutive windows share 49 of 50 samples, and a
stream of $n$ samples yields $n-49$ windows (stride = 50 recovers disjoint
tumbling windows; settable via env `DRIFT_STRIDE`). With overlap,
consecutive statistics are autocorrelated, so
false alarms are deduplicated into *events* with a cooldown of one full window
of fresh data, and the AIET target is restated in stride-steps so the
false-alarm rate **per unit time** is identical at every stride (one expected
false-alarm event per 370×50 samples). This aligns the nominal validation
operating point; it does not guarantee identical realised AIET after transfer
to held-out test identities.

**Three-way IC identity split.** The 6,000 class-0 identities are partitioned
per seed into 3,000 training, 1,500 validation, and 1,500 test identities.
The VAE, PCA bases, and fixed references use only the training identities;
detector thresholds use only a long epoch-cycled validation stream; and the
drift-free verification and drift streams use only the untouched test
identities. The training partition is deliberately the same 3,000-identity
side used for fitting in the original v6 run. Compatible VAE/PCA/reference
artifacts may therefore be reused, but original thresholds and test results
are never reused.

**What is shared vs. per-seed.** Shared across seeds: only the frozen
extractors (§0). Per seed: calibration-stream index arrays, PCA basis per
block, the VAE and its threshold, calibrated thresholds, and all 16 test
streams. Seed state is pickled under
`runs_v6_train_validation_test_20seeds/{extractor}/seed{N}/`; aggregates,
comparison tables, and figures go to the corresponding `aggregate/` folder.
The original `runs_v6/` directory is read-only input only for compatible
training-fitted artifacts; no earlier thresholds, evaluations, or comparison
summaries enter the reported benchmark. Cached artifacts are reused on
re-run and recomputed automatically if incompatible (stride change, missing
methods); `Config.force_recompute=True` forces a redo.

**Runtime.** Full run at the default stride 1 on Apple-Silicon MPS: no
extractor training; roughly 15–30 min per seed (96 px extraction; the
calibration grid contains 554,951 overlapping scores per block; MMD uses an exact streaming
decomposition so the cost stays linear in stream length) → several hours for
20 primary seeds; run it overnight if the original training artifacts are not
available. Set env `DRIFT_QUICK=1` before
starting the kernel for a scaled-down smoke run (1 seed, AIET target 50, short
streams) that exercises every code path in a few minutes.


In [ ]:
from __future__ import annotations

import io, math, os, pickle, time, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.lib.stride_tricks import sliding_window_view
from PIL import Image
from scipy import special, stats as sstats
from sklearn.decomposition import PCA
from torchvision import datasets, models
from torchvision.transforms import functional as TF
import matplotlib.pyplot as plt

%matplotlib inline
warnings.filterwarnings("ignore")

QUICK = os.environ.get("DRIFT_QUICK", "0") == "1"
DEVICE = torch.device("mps" if torch.backends.mps.is_available()
                      else ("cuda" if torch.cuda.is_available() else "cpu"))


@dataclass
class Config:
    # Calibration target: AIET = 370 window equivalents. The target_arl0
    # variable name is retained for cache compatibility; with overlap and
    # stateful charts this is not a classical independent-window ARL0.
    # Fixed 20-seed range used throughout the reported study.
    seeds: Tuple[int, ...] = (42,) if QUICK else tuple(range(42, 62))
    window: int = 50                                  # samples per window
    stride: int = int(os.environ.get("DRIFT_STRIDE", "1"))
    # window-start spacing in samples: default 1 = maximally sliding windows
    # (every sample starts a new window; consecutive windows share 49 of 50
    # samples; AIET handled via event dedup below); 50 = disjoint tumbling
    target_arl0: float = 50.0 if QUICK else 370.0     # in windows
    arl0_tol: float = 0.10                            # +/-10% acceptance band
    cal_arl0_mult: int = 20 if QUICK else 30          # cal length = mult * target_arl0 windows

    ic_class: int = 0                                 # in-control class (airplane)
    # Three identity-disjoint partitions of the 6,000 class-0 images.
    # Representation/reference fitting: 3,000; threshold validation: 1,500;
    # final null/drift evaluation: 1,500.
    ic_train_frac: float = 0.50
    ic_validation_frac: float = 0.25
    pca_dim: int = 12 if QUICK else 20                # retained PCs per block
    # PCA retention criterion: "fixed" keeps pca_dim PCs in every block
    # (reported benchmark: identical dimensionality -> Bonferroni penalty and
    # covariance-estimation burden matched across blocks); "evr" keeps the
    # smallest d reaching pca_evr cumulative explained variance (per block).
    pca_criterion: str = "fixed"
    pca_evr: float = 0.7
    pca_fit_n: int = 8000 if QUICK else 50000         # cal-stream samples used to fit PCA
    ref_n: int = 300 if QUICK else 1000               # fixed IC reference sample size

    # test streams (units: 50-sample windows; every changepoint is aligned to
    # a 50-sample block boundary regardless of stride)
    n_windows_test: int = 60 if QUICK else 300
    cp_window: int = 30 if QUICK else 150             # first changepoint
    ramp_windows: int = 10 if QUICK else 50           # incremental/gradual transition width
    recur_half: int = 5 if QUICK else 25              # recurring ON/OFF half-period

    # OOC contamination: injection-rate ceiling. The legacy setup identifier
    # remains 'label' in cached files; injections
    # are drawn uniformly across classes 1-9 (configurable via the pool below).
    ooc_rate_max: float = 0.5

    # drift-magnitude sweep: each level s < 1 adds severity-scaled variants of
    # the four SUDDEN setups (transform severity / OOC rate scaled by s); the
    # canonical 16 setups are the s = 1.0 members and are not duplicated.
    severity_levels: Tuple[float, ...] = (0.5, 1.0) if QUICK else (0.25, 0.5, 1.0)

    # transform severity maps, s in [0,1] (documented defaults; s=0 -> identity)
    jpeg_q_hi: int = 95                               # quality = q_hi - (q_hi - q_lo)*s
    jpeg_q_lo: int = 10
    stretch_max: float = 0.5                          # width scale = 1 + 0.5*s
    saturation_max: float = 1.5                       # factor    = 1 + 1.5*s

    hist_bins: int = 10                               # KL-Hist / Hellinger-fixed / PSI
    mmd_ref_subsample: int = 500                      # median-heuristic bandwidth sample
    cusum_k: float = 0.5                              # CUSUM allowance (sigma units)
    ewma_lambda: float = 0.2                          # EWMA smoothing weight

    # frozen extractor variants: NO CIFAR-10 training anywhere.
    # primary = off-the-shelf ImageNet weights; 'random' = untrained control.
    primary_extractor: str = "imagenet"
    # Disabled here so the requested comparison contains exactly 20 full seeds.
    compare_random_extractor: bool = False
    random_extractor_seeds: Tuple[int, ...] = ()
    extract_size: int = 96                            # 32px inputs upsampled for the ImageNet stem
    extract_batch: int = 1024

    # per-seed VAE reconstruction-error detector (+ latent feature block)
    vae_latent: int = 64
    vae_epochs: int = 2 if QUICK else 20
    vae_batch: int = 256

    # Optional unreported alarm-triggered updating module
    adapt_monitor: Tuple[str, str] = ("layer3", "KL-Gauss")
    adapt_confirm_events: int = 2 if QUICK else 3     # alarm events to confirm drift
    adapt_confirm_horizon: int = 5 if QUICK else 10   # ... within this many windows
    adapt_buffer_windows: int = 10 if QUICK else 40   # data collected per retraining
    adapt_periodic_T: Tuple[int, ...] = (20,) if QUICK else (50, 100, 200)
    run_adaptation: bool = False                      # disabled; not part of the reported benchmark

    # per-seed realised-AIET verification: a dedicated drift-free stream drawn from the
    # HELD-OUT identities, length = null_stream_mult * target_arl0 windows
    # (~null_stream_mult expected alarm events per (block, method) per seed).
    # Safe to disable (nothing downstream depends on it); costs ~one
    # calibration-length pass per seed when enabled.
    run_null_stream: bool = True
    null_stream_mult: int = 20 if QUICK else 30

    # Optional unreported fixed-d PCA sweep (disabled by default)
    run_pca_ablation: bool = os.environ.get("DRIFT_ABLATION", "0") == "1"  # sweep off by default
    # uniform fixed-d sweep: the main run's pca_dim is the middle point and is
    # reused (not recomputed); only these extra dims are run. All below the
    # window size (50) so every detector, incl. covariance-based KL-Gauss,
    # stays well-conditioned, and all blocks share one d so comparisons keep
    # dimension parity.
    ablation_dims: Tuple[int, ...] = (8, 16) if QUICK else (12, 32)
    ablation_seeds: Tuple[int, ...] = (42,)

    data_root: str = "data"
    out_root: str = ("runs_v6_tvt_20seeds_quick" if QUICK
                     else "runs_v6_train_validation_test_20seeds")
    baseline_root: str = "runs_v6"                  # read-only comparison/reuse source
    reuse_baseline_train_artifacts: bool = True
    force_recompute: bool = False

    @property
    def cal_windows(self) -> int:
        # Number of 50-sample window-equivalents used to set stream exposure;
        # this is not the number of overlapping scores when stride < window.
        return int(round(self.cal_arl0_mult * self.target_arl0))

    @property
    def cal_samples(self) -> int:
        return self.cal_windows * self.window

    @property
    def cooldown(self) -> int:
        # alarm-event dedup: one full window of fresh data (1 at stride=window,
        # i.e. no dedup for disjoint tumbling windows)
        return max(1, math.ceil(self.window / self.stride))

    @property
    def target_arl0_steps(self) -> float:
        # AIET target in stride-steps: same expected time between false-alarm
        # events (target_arl0 * window samples) at every stride
        return self.target_arl0 * self.window / self.stride


CFG = Config()
BLOCKS = ("layer1", "layer2", "layer3", "layer4")
FEATURE_BLOCKS = BLOCKS + ("vae_latent",)   # + VAE posterior-mean latents as a
                                            #   5th feature block (same PCA +
                                            #   divergence pipeline)
# Legacy cache labels: KSWIN = KS-fixed; HDDDM = Hellinger-fixed.
METHODS = ("KSWIN", "KL-Gauss", "KL-Hist", "MMD", "HDDDM", "PSI",
           "T2", "SPE", "Frechet", "CUSUM", "EWMA", "CUSUM-R")
ALL_METHODS = METHODS + ("VAE",)          # VAE operates on pixel space ("pixel" block)
PATTERNS = ("sudden", "incremental", "gradual", "recurring")
TRANSFORMS = ("jpeg", "stretch", "saturation")
BASE_SETUPS = ([f"concept-{p}-{t}" for p in PATTERNS for t in TRANSFORMS]
               + [f"label-{p}" for p in PATTERNS])
# severity-scaled variants of the sudden setups ("<setup>@<s>"); the canonical
# sudden setups ARE the s = 1.0 level, so only sub-1 levels add streams
EXTRA_SETUPS = [f"{s}@{sev:g}"
                for sev in sorted(set(CFG.severity_levels) - {1.0})
                for s in ([f"concept-sudden-{t}" for t in TRANSFORMS]
                          + ["label-sudden"])]
SETUPS = BASE_SETUPS + EXTRA_SETUPS
N_BASE = len(BASE_SETUPS)


def parse_setup(setup_id: str) -> Tuple[str, str, Optional[str], float]:
    core, _, sev = setup_id.partition("@")
    severity = float(sev) if sev else 1.0
    parts = core.split("-")
    if parts[0] == "concept":
        return "concept", parts[1], parts[2], severity
    return "label", parts[1], None, severity


# ---- per-seed RNG child registry -------------------------------------------
# Child indices are PINNED to the canonical-setup count so every artifact that
# existed before the severity sweep (base test streams, VAE training stream,
# identity split, null stream) reproduces bit-identically when EXTRA_SETUPS
# grows: 0 = representation-training stream, 1 = training-side reference,
# 2..2+N_BASE-1 = base test streams, then VAE / identity split / null stream,
# then severity-variant streams. Validation is appended after every legacy
# child so the original training/test RNG assignments remain unchanged.
CHILD_VAE = 2 + N_BASE
CHILD_SPLIT = 3 + N_BASE
CHILD_NULL = 4 + N_BASE
CHILD_VALIDATION = 5 + N_BASE + len(EXTRA_SETUPS)


def spawn_children(seed: int) -> list:
    return np.random.SeedSequence(seed).spawn(6 + N_BASE + len(EXTRA_SETUPS))


def child_for_setup(si: int) -> int:
    return 2 + si if si < N_BASE else 5 + si   # extras start at 5 + N_BASE


Path(CFG.out_root, "aggregate").mkdir(parents=True, exist_ok=True)
print(f"device={DEVICE}  QUICK={QUICK}")
print(f"window={CFG.window}, stride={CFG.stride} "
      f"({'tumbling' if CFG.stride >= CFG.window else 'overlapping sliding'}), "
      f"cooldown={CFG.cooldown} steps, AIET target={CFG.target_arl0:.0f} "
      f"window-equivalents "
      f"= {CFG.target_arl0_steps:.0f} steps")
print(f"IC identity split: train={CFG.ic_train_frac:.0%}, "
      f"validation={CFG.ic_validation_frac:.0%}, "
      f"test={1 - CFG.ic_train_frac - CFG.ic_validation_frac:.0%} "
      f"of class-{CFG.ic_class}")
print(f"{N_BASE} canonical drift setups + {len(EXTRA_SETUPS)} severity variants "
      f"(levels {CFG.severity_levels}); validation stream: {CFG.cal_windows} "
      f"window-equivalents = {CFG.cal_samples:,} samples, "
      f"{((CFG.cal_samples - CFG.window) // CFG.stride + 1):,} "
      f"overlapping scores per seed; "
      f"seeds={CFG.seeds}")


## §0 Shared frozen feature extractors — no CIFAR-10 training anywhere

The spec requires obtaining the extractor without the IC/OOC split as its
objective and explicitly allows "an ImageNet/self-supervised pretrained
backbone". This version takes the strongest form of that guarantee: **no part
of any extractor's weights is derived from CIFAR-10 at all.** Two frozen
variants:

- **`imagenet` (primary)** — off-the-shelf torchvision ResNet-18 with ImageNet
  weights, frozen (`requires_grad=False`, `eval()`). The 32×32 inputs are
  bilinearly upsampled to `extract_size` = 96 px and ImageNet-normalized (the
  CIFAR 3×3-stem design of earlier versions existed to suit CIFAR *training*;
  with pretrained frozen weights, the standard stem + upsampling is the
  appropriate interface). These weights have **never seen a CIFAR image**.
- **`random` (control)** — the identical architecture at its random
  initialization (fixed by `torch.manual_seed(0)`), never trained on anything.
  Only its BatchNorm *running statistics* are settled by forwarding ~10k
  unlabeled CIFAR images once (statistics normalization, not weight training —
  without it, eval-mode activations are arbitrarily mis-scaled). This variant
  isolates "deep architecture as structured random projection" from "learned
  representations", and doubles as a leakage control: wherever random ≈
  imagenet, the result cannot depend on anything the weights learned. It runs
  on `random_extractor_seeds`; the primary runs on all seeds.

**Feature taps** (both variants). A forward hook on each residual stage
(`layer1..layer4`) feeds a *parallel pooling branch*: global average-pool and
global max-pool of the block output, concatenated into one probe vector per
block per image (dims 128 / 256 / 512 / 1024). The hook only *reads* the
tensor — it returns nothing, so the activation flowing to the next block is
untouched. Both models are frozen and reused **identically across every seed**.


In [ ]:
_IN_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
_IN_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def to_unit_tensor(u8: np.ndarray) -> torch.Tensor:
    """(B,32,32,3) uint8 -> (B,3,32,32) float32 in [0,1] on DEVICE."""
    x = torch.from_numpy(np.ascontiguousarray(u8)).to(DEVICE)
    return x.permute(0, 3, 1, 2).contiguous().float().div_(255.0)


def to_backbone_input(u8: np.ndarray) -> torch.Tensor:
    """uint8 batch -> upsampled (extract_size px), ImageNet-normalized tensor
    for the frozen backbones."""
    x = F.interpolate(to_unit_tensor(u8), size=CFG.extract_size,
                      mode="bilinear", align_corners=False)
    return (x - _IN_MEAN) / _IN_STD


def augment_batch(u8: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Light stochastic augmentation: reflect pad-4 random crop + horizontal
    flip. Fully driven by `rng`, so a stream is reconstructible by replaying
    the same draw sequence from the pickled seed."""
    b = u8.shape[0]
    padded = np.pad(u8, ((0, 0), (4, 4), (4, 4), (0, 0)), mode="reflect")
    dy = rng.integers(0, 9, size=b)
    dx = rng.integers(0, 9, size=b)
    rows = (dy[:, None] + np.arange(32))[:, :, None]
    cols = (dx[:, None] + np.arange(32))[:, None, :]
    out = padded[np.arange(b)[:, None, None], rows, cols, :]
    flip = rng.random(b) < 0.5
    out[flip] = out[flip][:, :, ::-1, :]
    return out


print("Loading CIFAR-10 ...")
_tr = datasets.CIFAR10(CFG.data_root, train=True, download=True)
_te = datasets.CIFAR10(CFG.data_root, train=False, download=True)
IMAGES = np.concatenate([_tr.data, _te.data])            # (60000,32,32,3) uint8
LABELS = np.concatenate([np.asarray(_tr.targets), np.asarray(_te.targets)])
print(f"pool: {IMAGES.shape[0]:,} images; IC class {CFG.ic_class}: "
      f"{(LABELS == CFG.ic_class).sum():,} images")


def build_backbone(variant: str) -> nn.Module:
    """Frozen ResNet-18 backbones — no CIFAR-10 training anywhere.

    'imagenet': off-the-shelf ImageNet weights (never saw a CIFAR image).
    'random':  the same architecture at its seeded random initialization,
               never trained; only the BatchNorm running statistics are
               settled by forwarding ~10k unlabeled CIFAR images once
               (statistics normalization, not weight training).
    """
    if variant == "imagenet":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    elif variant == "random":
        torch.manual_seed(0)
        model = models.resnet18(weights=None)
        model = model.to(DEVICE).train()
        with torch.no_grad():
            for i in range(0, 10240, 512):
                _ = model(to_backbone_input(_tr.data[i:i + 512]))
    else:
        raise ValueError(variant)
    model = model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model


In [ ]:
class BlockProbeExtractor:
    """Frozen backbone with read-only forward hooks on each residual stage.

    Each hook only READS the block output for a parallel pooling branch
    (global avg-pool + global max-pool, concatenated); it returns nothing, so
    the tensor flowing to the next block is untouched. One probe vector per
    block per image.
    """

    def __init__(self, model: nn.Module):
        self.model = model.to(DEVICE).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self._buf: Dict[str, torch.Tensor] = {}
        for name in BLOCKS:
            getattr(self.model, name).register_forward_hook(self._make_hook(name))

    def _make_hook(self, name):
        def hook(_module, _inputs, out):
            self._buf[name] = torch.cat(
                [out.mean(dim=(2, 3)), out.amax(dim=(2, 3))], dim=1)
            # no return value: block output continues through the network unmodified
        return hook

    @torch.no_grad()
    def extract(self, u8: np.ndarray, batch: Optional[int] = None) -> Dict[str, np.ndarray]:
        batch = batch or CFG.extract_batch
        outs: Dict[str, List[np.ndarray]] = {n: [] for n in BLOCKS}
        for i in range(0, len(u8), batch):
            self._buf.clear()
            _ = self.model(to_backbone_input(u8[i:i + batch]))
            for n in BLOCKS:
                outs[n].append(self._buf[n].float().cpu().numpy())
        return {n: np.concatenate(v) for n, v in outs.items()}


EXTRACTORS = {"imagenet": BlockProbeExtractor(build_backbone("imagenet"))}
if CFG.compare_random_extractor:
    EXTRACTORS["random"] = BlockProbeExtractor(build_backbone("random"))


def variant_seeds(variant: str, cfg: Config) -> Tuple[int, ...]:
    return cfg.seeds if variant == cfg.primary_extractor else cfg.random_extractor_seeds


for _v, _ex in EXTRACTORS.items():
    _dims = {n: a.shape[1] for n, a in _ex.extract(IMAGES[:64]).items()}
    print(f"{_v}: probe dims per block: {_dims}")


## Stream construction & the four temporal-pattern definitions

**Epoch-cycled shuffling (§1a).** Every IC draw comes from the ~6,000-image
class-0 pool: the pool is shuffled once per epoch, samples are drawn sequentially
with no repeats within an epoch, and the pool is reshuffled only at epoch
boundaries. Light stochastic augmentation (reflect-pad-4 random crop +
horizontal flip) is applied on **every** draw, so repeated pool indices never
yield identical images. All randomness (shuffle order + augmentation) comes from
a per-stream `numpy` Generator, and only the epoch-shuffle index log + the
RNG seed are pickled — the whole stream is reconstructible from those alone.

**RNG isolation.** Each seed's `SeedSequence(seed)` is spawned into independent
children with **pinned indices** (registry in §0's config cell): child 0 =
training PCA stream, child 1 = training reference sample, children 2…17 = the 16
canonical test streams (each of which spawns three sub-generators: IC sampler,
OOC sampler, mixture decisions), child 18 = the VAE training stream, child 19
= the IC identity-split permutation, child 20 = the §5a null stream, children
21+ = the severity-variant streams, followed by a new validation-stream child.
Because the indices are pinned rather than
derived from `len(SETUPS)`, adding severity levels changes **nothing** about
the canonical streams, the VAE, or the identity split. Training, validation,
and test partitions share no image identities.

**Temporal patterns** (drift parameter $s_w \in [0,1]$ per 50-sample stream
block $w$; all changepoints and transition widths are in these block units and
aligned to block boundaries; $c$ = changepoint, $\rho$ = `ramp_windows`,
$h$ = `recur_half`):

- **Sudden** — $s_w = \mathbf{1}\{w \ge c\}$.
- **Incremental** — *parameter interpolation*: $s_w$ rises linearly,
  $s_w = \min(1, (w - c + 1)/\rho)$ for $w \ge c$; every sample in block $w$ is
  transformed at intermediate severity $s_w$.
- **Gradual** — *probabilistic source/target mixture*: each sample is drawn from
  the **fully drifted** target (severity 1) with probability $\pi_w$ and from the
  clean source otherwise, with $\pi_w$ following the same ramp. Distinct from
  incremental: windows mix pure-source and pure-target samples instead of
  containing uniformly half-drifted ones.
- **Recurring** — alternating ON/OFF blocks of $h$ windows starting at $c$
  ($s=1$ during ON, $s=0$ during OFF); **each ON onset is a changepoint**, so a
  stream contains multiple drift occurrences.

**Image-transformation streams (12)** apply the pattern to a transform severity
(JPEG artifacting / stretch-aspect warp / saturation shift; severity maps in the
transform cell, $s{=}0$ = identity). **OOC-contamination streams (4)** apply the same
patterns to the **OOC-injection rate**, scaled by `ooc_rate_max = 0.5`
(documented default, configurable); injections are drawn uniformly across
classes 1–9. For OOC contamination, *gradual* switches **whole blocks** between the
source regime (rate 0) and the target regime (rate `ooc_rate_max`) with
probability $\pi_w$ — a sample-level mixture would be distributionally identical
to incremental rate interpolation, so the block-level regime mixture is what
keeps the two patterns distinct. Image transformation and OOC contamination are never compounded in
the same stream.

Note the stream *data* is constructed in 50-sample blocks independent of the
detector stride; only the detector's window grid changes with `Config.stride`.


In [ ]:
class EpochCycledSampler:
    """Epoch-cycled shuffling over a fixed image pool (see markdown above).

    Shuffles the pool once per epoch, draws sequentially without repeats
    within an epoch, reshuffles only at epoch boundaries, and applies the
    light stochastic augmentation on every draw. All randomness comes from a
    single per-stream Generator, so the stream is reconstructible from the
    pickled SeedSequence + index log alone.
    """

    def __init__(self, pool_ids: np.ndarray, seed_seq: np.random.SeedSequence):
        self.pool_ids = np.asarray(pool_ids)
        self.seed_seq = seed_seq
        self.rng = np.random.default_rng(seed_seq)
        self._order: Optional[np.ndarray] = None
        self._pos = 0
        self._consumed: List[np.ndarray] = []

    def _next_indices(self, n: int) -> np.ndarray:
        out = np.empty(n, dtype=np.int64)
        filled = 0
        while filled < n:
            if self._order is None or self._pos >= len(self._order):
                self._order = self.rng.permutation(len(self.pool_ids))
                self._pos = 0
            take = min(n - filled, len(self._order) - self._pos)
            out[filled:filled + take] = self.pool_ids[
                self._order[self._pos:self._pos + take]]
            self._pos += take
            filled += take
        self._consumed.append(out)
        return out

    def draw(self, n: int) -> Tuple[np.ndarray, np.ndarray]:
        ids = self._next_indices(n)
        return augment_batch(IMAGES[ids], self.rng), ids

    def consumed_indices(self) -> np.ndarray:
        return (np.concatenate(self._consumed) if self._consumed
                else np.empty(0, dtype=np.int64))


def ic_pools(seed: int, cfg: Config) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return identity-disjoint (training, validation, test) IC pools.

    For the default 50/25/25 split, training is the final 3,000 identities of
    the same dedicated permutation used by v6. It is therefore exactly the
    original calibration-side pool. Validation and test split the original
    3,000-identity holdout, with test kept first in the permutation. The split
    child is unchanged, so all partitions are deterministic per seed."""
    pool = np.flatnonzero(LABELS == cfg.ic_class)
    if not (0 < cfg.ic_train_frac < 1 and 0 < cfg.ic_validation_frac < 1):
        raise ValueError("training and validation fractions must be in (0, 1)")
    if cfg.ic_train_frac + cfg.ic_validation_frac >= 1:
        raise ValueError("training + validation fractions must leave a test partition")
    child = spawn_children(seed)[CHILD_SPLIT]
    perm = np.random.default_rng(child).permutation(len(pool))
    n_train = int(round(len(pool) * cfg.ic_train_frac))
    n_validation = int(round(len(pool) * cfg.ic_validation_frac))
    n_test = len(pool) - n_train - n_validation
    test_ids = pool[perm[:n_test]]
    validation_ids = pool[perm[n_test:n_test + n_validation]]
    train_ids = pool[perm[n_test + n_validation:]]
    assert not (set(train_ids) & set(validation_ids)
                or set(train_ids) & set(test_ids)
                or set(validation_ids) & set(test_ids))
    return train_ids, validation_ids, test_ids


In [ ]:
def jpeg_batch(u8: np.ndarray, quality: int) -> np.ndarray:
    out = np.empty_like(u8)
    for i, img in enumerate(u8):
        buf = io.BytesIO()
        Image.fromarray(img).save(buf, format="JPEG", quality=int(quality))
        buf.seek(0)
        out[i] = np.asarray(Image.open(buf).convert("RGB"))
    return out


def stretch_batch(u8: np.ndarray, factor: float) -> np.ndarray:
    w = int(round(32 * factor))
    if w <= 32:
        return u8
    t = torch.from_numpy(np.ascontiguousarray(u8)).permute(0, 3, 1, 2).float()
    t = F.interpolate(t, size=(32, w), mode="bilinear", align_corners=False)
    left = (w - 32) // 2
    t = t[:, :, :, left:left + 32]
    return t.round().clamp(0, 255).byte().permute(0, 2, 3, 1).numpy()


def saturation_batch(u8: np.ndarray, factor: float) -> np.ndarray:
    t = torch.from_numpy(np.ascontiguousarray(u8)).permute(0, 3, 1, 2).float().div(255.0)
    t = TF.adjust_saturation(t, factor)
    return t.mul(255.0).round().clamp(0, 255).byte().permute(0, 2, 3, 1).numpy()


def apply_transform(name: str, u8: np.ndarray, s: float, cfg: Config) -> np.ndarray:
    """Severity maps (documented defaults): s=0 means identity (caller skips);
    jpeg: quality 95 -> 10; stretch: width scale 1 -> 1.5 then center-crop;
    saturation: factor 1 -> 2.5."""
    if s <= 0:
        return u8
    if name == "jpeg":
        q = int(round(cfg.jpeg_q_hi - (cfg.jpeg_q_hi - cfg.jpeg_q_lo) * s))
        return jpeg_batch(u8, q)
    if name == "stretch":
        return stretch_batch(u8, 1.0 + cfg.stretch_max * s)
    if name == "saturation":
        return saturation_batch(u8, 1.0 + cfg.saturation_max * s)
    raise ValueError(name)


_demo = IMAGES[np.flatnonzero(LABELS == CFG.ic_class)[3]][None]
fig, axes = plt.subplots(3, 5, figsize=(10, 6.5))
for r, t in enumerate(TRANSFORMS):
    for c, s in enumerate((0.0, 0.25, 0.5, 0.75, 1.0)):
        axes[r, c].imshow(apply_transform(t, _demo.copy(), s, CFG)[0])
        axes[r, c].set_axis_off()
        axes[r, c].set_title(f"{t} s={s:g}", fontsize=8)
fig.suptitle("Transform severity maps (s=0 -> identity)")
fig.tight_layout()
fig.savefig(Path(CFG.out_root, "aggregate", "transform_severities.png"), dpi=120)
plt.show()


In [ ]:
def temporal_schedule(pattern: str, cfg: Config):
    """Per-block drift-parameter schedule s_w in [0,1], plus the changepoints
    (blocks where a drift episode begins) and the drifted segments over which
    detection delay is measured. All boundaries are 50-sample-block-aligned."""
    n, cp, ramp, half = (cfg.n_windows_test, cfg.cp_window,
                         cfg.ramp_windows, cfg.recur_half)
    s = np.zeros(n)
    if pattern == "sudden":
        s[cp:] = 1.0
        return s, [cp], [(cp, n)]
    if pattern in ("incremental", "gradual"):
        s[cp:cp + ramp] = np.arange(1, ramp + 1) / ramp
        s[cp + ramp:] = 1.0
        return s, [cp], [(cp, n)]
    if pattern == "recurring":
        cps, segs = [], []
        t, on = cp, True
        while t < n:
            if on:
                e = min(t + half, n)
                s[t:e] = 1.0
                cps.append(t)
                segs.append((t, e))
            t += half
            on = not on
        return s, cps, segs
    raise ValueError(pattern)


def build_test_stream(setup_id: str, cfg: Config,
                      ic_sampler: EpochCycledSampler,
                      ooc_sampler: EpochCycledSampler,
                      mix_rng: np.random.Generator):
    """Materialize one drift stream (uint8 images) + metadata.

    concept: s_w drives the transform severity (sudden/incremental/recurring)
             or the per-sample source/target mixture probability (gradual,
             target = full severity).
    label:   s_w drives the OOC injection rate scaled by ooc_rate_max;
             gradual switches whole blocks between regimes (see markdown).
    severity: "<setup>@<s>" variants scale the whole schedule by s (drift
             magnitude); only sudden setups have such variants, so the scaling
             is exact for both the transform-severity and OOC-rate readings.
    """
    kind, pattern, transform, severity = parse_setup(setup_id)
    s, cps, segs = temporal_schedule(pattern, cfg)
    s = s * severity
    w = cfg.window
    imgs = np.empty((len(s) * w, 32, 32, 3), dtype=np.uint8)
    for wi, sw in enumerate(s):
        batch, _ = ic_sampler.draw(w)
        if kind == "concept" and sw > 0:
            if pattern == "gradual":
                mask = mix_rng.random(w) < sw
                if mask.any():
                    batch[mask] = apply_transform(transform, batch[mask], 1.0, cfg)
            else:
                batch = apply_transform(transform, batch, float(sw), cfg)
        elif kind == "label" and sw > 0:
            if pattern == "gradual":
                rate = cfg.ooc_rate_max if mix_rng.random() < sw else 0.0
            else:
                rate = float(sw) * cfg.ooc_rate_max
            if rate > 0:
                mask = mix_rng.random(w) < rate
                k = int(mask.sum())
                if k:
                    ooc_batch, _ = ooc_sampler.draw(k)
                    batch[mask] = ooc_batch
        imgs[wi * w:(wi + 1) * w] = batch
    meta = {"setup": setup_id, "kind": kind, "pattern": pattern,
            "transform": transform or "ooc", "severity": severity,
            "schedule": s, "changepoints": cps, "on_segments": segs,
            "n_windows": len(s), "cp": cfg.cp_window}
    return imgs, meta


for _p in PATTERNS:
    _s, _cps, _segs = temporal_schedule(_p, CFG)
    print(f"{_p:12s} changepoints={_cps}  drifted segments={_segs}")


## Divergence methods & estimator choices

The retained distributional methods consume the **same PCA-projected probe vectors**
(per block) and the same fixed-reference / strided-window geometry, and all
detectors are calibrated by the identical §1c protocol. This matches the nominal
validation AIET target but does not force the realised test AIETs to be equal. Alarm
direction: statistic **above** threshold, except KS-fixed (stored as `KSWIN`) which alarms when its
min p-value falls **below** threshold.

- **KS-fixed** (legacy cache label `KSWIN`) — an exact two-sample KS statistic
  between the fixed reference and current window is computed per retained PC,
  converted to an asymptotic $p$-value, and combined as $\min_j p_j$. Its single
  cutoff is selected empirically from the joint validation sequence, so there is
  no separate Bonferroni test at $\alpha/d$. This is a fixed-reference marginal
  KS monitor, not a reproduction of the moving-reference KSWIN algorithm.
- **KL-Gauss** — closed-form Gaussian KL $\mathrm{KL}(\mathcal N_{\text{win}}\,\|\,
  \mathcal N_{\text{ref}})$ between window-fitted Gaussians. Covariances on both
  sides use **Ledoit–Wolf shrinkage** (vectorized batch implementation, verified
  against `sklearn` below), which is what allows retaining $d=20$ PCs at window
  size 50 rather than the naive dims ≪ 50 rule.
- **KL-Hist** — secondary/sanity-check KL estimator: per-PC histograms
  (reference-decile bins, open-ended outer bins), **Laplace-smoothed**, per-PC
  $\mathrm{KL}(q_{\text{win}} \| p_{\text{ref}})$ averaged over PCs. Reported
  alongside KL-Gauss everywhere; their agreement is quantified in the
  aggregation section.
- **MMD** — unbiased squared Maximum Mean Discrepancy with an RBF kernel
  (median-heuristic bandwidth on a fixed reference subsample). Handles the
  multivariate PCA features natively — no per-dimension decomposition.
- **Hellinger-fixed** (legacy cache label `HDDDM`) — Hellinger distance per PC
  between window and fixed-reference histograms
  (same bins as KL-Hist, unsmoothed — Hellinger needs no smoothing), aggregated
  family-wise via the **max over PCs** (the Bonferroni analog for a
  non-p-value statistic). Natural baseline against PCA+KL: same multivariate
  problem, different aggregation rule. It uses the distance underlying HDDDM,
  but not HDDDM's moving reference or adaptive distance-change threshold.
- **PSI** — industry-standard Population Stability Index per PC
  (Laplace-smoothed, reference-decile bins), averaged over PCs.
- **T²** — Hotelling's $T^2$ on the window mean:
  $T^2_w = m\,(\bar x_w - \mu_{\text{ref}})' \Sigma_{\text{ref}}^{-1}
  (\bar x_w - \mu_{\text{ref}})$ with the reference Ledoit–Wolf covariance.
  The canonical multivariate SPM location chart (De Ketelaere, Hubert &
  Schmitt, JQT 2015): pure mean-shift sensitivity, no covariance term —
  the paradigm complement to KL-Gauss, which mixes both.
- **SPE** — the squared prediction error (Q chart), the standard companion of
  $T^2$ in PCA-based process monitoring: per-sample squared residual off the
  retained PCA basis, $\mathrm{SPE}(x) = \lVert x_c\rVert^2 -
  \lVert P x_c\rVert^2$ (orthonormal loadings), averaged per window. All other
  PCA methods are *structurally blind* to drift that lives in the discarded
  residual subspace; SPE monitors exactly that subspace, closing the classic
  $T^2$/SPE decomposition over each feature block.
- **Frechet** — closed-form Fréchet (Wasserstein-2) distance between the
  window and reference Gaussians, $\lVert\mu_w-\mu_r\rVert^2 +
  \mathrm{tr}\!\left(\Sigma_w + \Sigma_r - 2(\Sigma_r^{1/2}\Sigma_w
  \Sigma_r^{1/2})^{1/2}\right)$, on the same Ledoit–Wolf moments as KL-Gauss.
  This is the statistic behind FID and DriftLens's Fréchet Drift Distance
  (Greco et al. 2024), included for direct comparability with that line of
  work; unlike KL it is a true metric and stays finite under support mismatch.
- **EWMA** — the retained sequential mean chart. Unlike the window-isolated
  methods above, it accumulates evidence across windows. It charts the standardized per-PC
  window means $u_{t,j} = (\bar x_{t,j} - \mu_{\text{ref},j})
  /(\sigma_{\text{ref},j}/\sqrt{m})$ on the same stride grid. With $\lambda=0.2$,
  its statistic is $\|z_t\|^2 (2-\lambda)/\lambda$. This is a mean-shift
  MEWMA-style chart and does not target pure covariance changes. The strong
  dependence created by 49/50 window overlap invalidates textbook independent-input
  control-limit formulas; the empirical validation threshold matches the nominal
  AIET operating point but does not make the independence assumption true.
- **Legacy CUSUM variants (excluded from the manuscript)** — the raw notebook
  also computes no-reset and reset-on-alarm CUSUM variants. They are retained in
  cached outputs for auditability but are not part of the reported eleven-method
  comparison because their realised held-out false-alarm-event rates were not
  adequately matched.
- **VAE** — per-seed convolutional VAE (latent 64) trained on IC-only draws
  from a dedicated RNG continuation of that seed's pool scheme. Statistic =
  per-window mean of the per-sample pixel reconstruction MSE (deterministic
  encoding: decode the posterior mean). Its alarm threshold is *optimized in
  validation* per seed by the same empirical-AIET search as every other method.
  The target is one false-alarm event per 370 window equivalents, subject to
  held-out transfer variation. Appears
  in all tables as method `VAE`, block `pixel` (it bypasses the ResNet/PCA
  features entirely).

**VAE latent block.** In addition to the reconstruction-error statistic, the
VAE's *encoder* doubles as a fifth feature extractor: the posterior mean
$\mu(x)$ (64-dim) is treated exactly like a ResNet probe vector — per-seed PCA
fit on the training stream, projection to `pca_dim`, and all five divergence
methods calibrated against a fixed latent reference (block `vae_latent`). This
separates two claims the reconstruction statistic conflates: whether
*generative-model features* carry the drift signal vs. whether *reconstruction
error* is a good statistic — the decoder's blurriness can hide corruptions
(JPEG, warp) from the MSE even when the encoder's latents shift.


In [ ]:
def ledoit_wolf_batched(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Vectorized Ledoit-Wolf shrinkage covariance for a batch of windows.
    x: (nw, m, d). Returns means (nw, d) and covariances (nw, d, d).
    Matches sklearn.covariance.LedoitWolf (normalization by m)."""
    nw, m, d = x.shape
    mu = x.mean(axis=1)
    xc = x - mu[:, None, :]
    s = xc.transpose(0, 2, 1) @ xc / m       # batched BLAS, = einsum nmi,nmj->nij
    tr = np.einsum("nii->n", s)
    mu_i = tr / d
    s_frob2 = np.einsum("nij,nij->n", s, s)
    d2 = s_frob2 - 2.0 * mu_i * tr + d * mu_i ** 2
    sq_norms = np.einsum("nmi,nmi->nm", xc, xc)
    b2 = np.minimum((np.mean(sq_norms ** 2, axis=1) - s_frob2) / m, d2)
    shrink = np.where(d2 > 1e-30, b2 / np.maximum(d2, 1e-30), 1.0)
    cov = (1.0 - shrink)[:, None, None] * s
    cov[:, np.arange(d), np.arange(d)] += (shrink * mu_i)[:, None]
    return mu, cov


def prepare_reference(ref: np.ndarray, cfg: Config) -> Dict:
    """Precompute everything the five methods need about the fixed IC
    reference sample. Deterministic given `ref` (the RBF bandwidth uses the
    first mmd_ref_subsample rows), so it is exactly reproducible at eval time."""
    ref = np.asarray(ref, dtype=np.float64)
    r, d = ref.shape
    mu_r, cov_r = ledoit_wolf_batched(ref[None])
    mu_r, cov_r = mu_r[0], cov_r[0]
    cov_r_inv = np.linalg.inv(cov_r)
    _, logdet_r = np.linalg.slogdet(cov_r)
    # symmetric PSD square root of the reference covariance (for Frechet)
    evals_r, evecs_r = np.linalg.eigh(cov_r)
    cov_r_sqrt = (evecs_r * np.sqrt(np.clip(evals_r, 0.0, None))) @ evecs_r.T
    # histogram bins: reference-decile edges, open-ended outer bins
    qs = np.linspace(0, 1, cfg.hist_bins + 1)[1:-1]
    edges = np.quantile(ref, qs, axis=0).T                # (d, bins-1)
    counts = np.empty((d, cfg.hist_bins))
    for j in range(d):
        ids = np.searchsorted(edges[j], ref[:, j], side="right")
        counts[j] = np.bincount(ids, minlength=cfg.hist_bins)
    p_raw = counts / r
    p_lap = (counts + 1.0) / (r + cfg.hist_bins)
    # RBF bandwidth: median heuristic on a fixed subsample
    sub = ref[:min(cfg.mmd_ref_subsample, r)]
    sq = ((sub[:, None, :] - sub[None, :, :]) ** 2).sum(-1)
    med = np.median(sq[np.triu_indices_from(sq, k=1)])
    gamma = 1.0 / max(med, 1e-12)
    ref_sq = (ref ** 2).sum(1)
    d_rr = ref_sq[:, None] + ref_sq[None, :] - 2.0 * ref @ ref.T
    k_rr = np.exp(-gamma * np.maximum(d_rr, 0.0))
    term_rr = (k_rr.sum() - r) / (r * (r - 1))
    return {"ref": ref, "sorted": np.sort(ref, axis=0), "mu": mu_r,
            "cov_inv": cov_r_inv, "logdet": logdet_r,
            "cov_tr": float(np.trace(cov_r)), "cov_sqrt": cov_r_sqrt,
            "edges": edges,
            "p_raw": p_raw, "p_lap": p_lap, "gamma": gamma,
            "ref_sq": ref_sq, "term_rr": term_rr,
            "mu_marg": ref.mean(axis=0), "sd_marg": ref.std(axis=0) + 1e-12}


def pca_spe(pca: PCA, raw: np.ndarray, proj: np.ndarray) -> np.ndarray:
    """Per-sample squared prediction error (Q statistic) off the retained PCA
    basis: ||x_c||^2 - ||P x_c||^2, exact for orthonormal loadings (sklearn
    PCA), clipped at 0 against floating-point cancellation. `proj` is the
    already-computed projection so the basis is applied exactly once."""
    xc = np.asarray(raw, dtype=np.float64) - pca.mean_
    return np.maximum(np.einsum("nd,nd->n", xc, xc)
                      - np.einsum("nk,nk->n",
                                  np.asarray(proj, dtype=np.float64), proj),
                      0.0)


# ---- strided windowing ------------------------------------------------------
def n_windows_for(n_samples: int, cfg: Config) -> int:
    return (n_samples - cfg.window) // cfg.stride + 1


def make_windows(x: np.ndarray, w: int, stride: int) -> np.ndarray:
    """(n_samples, d) -> (nw, w, d); window i covers samples
    [i*stride, i*stride + w). stride == w gives disjoint tumbling windows."""
    v = sliding_window_view(x, w, axis=0)[::stride]      # (nw, d, w)
    return np.ascontiguousarray(v.transpose(0, 2, 1))


def window_means(values: np.ndarray, cfg: Config) -> np.ndarray:
    """Per-window mean of a scalar per-sample sequence on the strided grid
    (used by the VAE reconstruction-error detector)."""
    c = np.concatenate(([0.0], np.cumsum(values, dtype=np.float64)))
    starts = np.arange(n_windows_for(len(values), cfg)) * cfg.stride
    return (c[starts + cfg.window] - c[starts]) / cfg.window


def ks_d_batched(ref_sorted: np.ndarray, wins: np.ndarray) -> np.ndarray:
    """Exact two-sample KS statistic per (window, dim).
    ref_sorted: (R, d), sorted per column; wins: (nw, m, d).
    D = max( max_i((i+1)/m - F_ref(w_(i))), max_i(F_ref(w_(i)^-) - i/m) )."""
    nw, m, d = wins.shape
    r = ref_sorted.shape[0]
    ws = np.sort(wins, axis=1)
    i_hi = np.arange(1, m + 1) / m
    i_lo = np.arange(m) / m
    d_stat = np.empty((nw, d))
    for j in range(d):
        right = np.searchsorted(ref_sorted[:, j], ws[:, :, j], side="right") / r
        left = np.searchsorted(ref_sorted[:, j], ws[:, :, j], side="left") / r
        d_stat[:, j] = np.maximum((i_hi[None, :] - right).max(axis=1),
                                  (left - i_lo[None, :]).max(axis=1))
    return d_stat


def kswin_minp_stats(ref_prep: Dict, wins: np.ndarray) -> np.ndarray:
    """Per-PC KS p-values (asymptotic Kolmogorov distribution), combined via
    the minimum. One cutoff for this joint score is calibrated empirically;
    no separate Bonferroni alpha/d rule is applied."""
    r = ref_prep["sorted"].shape[0]
    m = wins.shape[1]
    en = math.sqrt(r * m / (r + m))
    return special.kolmogorov(en * ks_d_batched(ref_prep["sorted"], wins)).min(axis=1)


def gauss_moment_stats(ref_prep: Dict, wins: np.ndarray) -> Dict[str, np.ndarray]:
    """The three Gaussian-moment statistics from ONE shared Ledoit-Wolf pass
    over the windows:
    - KL-Gauss: closed-form KL( N_window || N_reference );
    - T2: Hotelling's T^2 of the window mean against the reference moments;
    - Frechet: closed-form Frechet/Wasserstein-2 distance between the two
      Gaussians (trace cross-term via batched eigendecomposition of the
      symmetric PSD matrix S_r^{1/2} S_w S_r^{1/2})."""
    nw, m, d = wins.shape
    mu_w, cov_w = ledoit_wolf_batched(np.asarray(wins, dtype=np.float64))
    diff = mu_w - ref_prep["mu"][None, :]
    maha = np.einsum("ni,ij,nj->n", diff, ref_prep["cov_inv"], diff)
    tr_term = np.einsum("ij,nji->n", ref_prep["cov_inv"], cov_w)
    _, logdet_w = np.linalg.slogdet(cov_w)
    kl = 0.5 * (tr_term + maha - d + ref_prep["logdet"] - logdet_w)
    t2 = m * maha
    s_half = ref_prep["cov_sqrt"]
    cross_ev = np.linalg.eigvalsh(s_half @ cov_w @ s_half)
    fd2 = (np.einsum("ni,ni->n", diff, diff)
           + np.einsum("nii->n", cov_w) + ref_prep["cov_tr"]
           - 2.0 * np.sqrt(np.clip(cross_ev, 0.0, None)).sum(axis=1))
    return {"KL-Gauss": kl, "T2": t2, "Frechet": np.maximum(fd2, 0.0)}


def window_bin_counts(edges: np.ndarray, wins: np.ndarray, n_bins: int) -> np.ndarray:
    """(nw, d, n_bins) histogram counts per window per dim, fixed ref-quantile
    edges with open-ended outer bins."""
    nw, m, d = wins.shape
    counts = np.empty((nw, d, n_bins))
    offsets = (np.arange(nw) * n_bins)[:, None]
    for j in range(d):
        ids = np.searchsorted(edges[j], wins[:, :, j], side="right")
        counts[:, j, :] = np.bincount((ids + offsets).ravel(),
                                      minlength=nw * n_bins).reshape(nw, n_bins)
    return counts


def mmd_stats(ref_prep: Dict, wins: np.ndarray, chunk: int = 200) -> np.ndarray:
    """Unbiased squared MMD, RBF kernel (median-heuristic bandwidth), between
    the fixed reference and each window."""
    ref, gamma = ref_prep["ref"], ref_prep["gamma"]
    ref_sq, term_rr = ref_prep["ref_sq"], ref_prep["term_rr"]
    nw, m, _ = wins.shape
    wins = np.asarray(wins, dtype=np.float64)
    out = np.empty(nw)
    for i in range(0, nw, chunk):
        wc = wins[i:i + chunk]
        w_sq = np.einsum("cmd,cmd->cm", wc, wc)
        d_ww = w_sq[:, :, None] + w_sq[:, None, :] - 2.0 * np.einsum("cmd,cnd->cmn", wc, wc)
        term_ww = (np.exp(-gamma * np.maximum(d_ww, 0.0)).sum(axis=(1, 2)) - m) / (m * (m - 1))
        d_rw = w_sq[:, :, None] + ref_sq[None, None, :] - 2.0 * np.einsum("cmd,rd->cmr", wc, ref)
        term_rw = np.exp(-gamma * np.maximum(d_rw, 0.0)).mean(axis=(1, 2))
        out[i:i + len(wc)] = term_rr + term_ww - 2.0 * term_rw
    return out


def mmd_stats_stream(ref_prep: Dict, proj: np.ndarray, cfg: Config,
                     chunk: int = 20000) -> np.ndarray:
    """Exact unbiased MMD^2 per window in O(n*R + n*m) via a per-sample
    decomposition (algebraically identical to mmd_stats — asserted below):
    the ref-window cross term is a rolling mean of per-sample kernel means
    phi_t = mean_r k(x_t, r), and the within-window pair sum is assembled
    from banded kernels k(x_t, x_{t+d}), d = 1..m-1, via cumulative sums.
    Required at small strides, where per-window evaluation would cost
    O(n_windows * m * R)."""
    ref, gamma = ref_prep["ref"], ref_prep["gamma"]
    ref_sq, term_rr = ref_prep["ref_sq"], ref_prep["term_rr"]
    x = np.asarray(proj, dtype=np.float64)
    n = len(x)
    m = cfg.window
    x_sq = np.einsum("nd,nd->n", x, x)
    phi = np.empty(n)
    for i in range(0, n, chunk):
        xc = x[i:i + chunk]
        d_rw = x_sq[i:i + chunk, None] + ref_sq[None, :] - 2.0 * xc @ ref.T
        phi[i:i + len(xc)] = np.exp(-gamma * np.maximum(d_rw, 0.0)).mean(axis=1)
    starts = np.arange(n_windows_for(n, cfg)) * cfg.stride
    cphi = np.concatenate(([0.0], np.cumsum(phi)))
    term_rw = (cphi[starts + m] - cphi[starts]) / m
    pair_sum = np.zeros(len(starts))
    for d in range(1, m):
        band = np.exp(-gamma * np.maximum(
            x_sq[:-d] + x_sq[d:] - 2.0 * np.einsum("nd,nd->n", x[:-d], x[d:]), 0.0))
        cb = np.concatenate(([0.0], np.cumsum(band)))
        pair_sum += cb[starts + m - d] - cb[starts]
    term_ww = 2.0 * pair_sum / (m * (m - 1))
    return term_rr + term_ww - 2.0 * term_rw


def _standardized_window_means(ref_prep: Dict, proj: np.ndarray,
                               cfg: Config) -> np.ndarray:
    """(nw, d) per-PC window means on the stride grid, standardized by the
    reference marginals: u = (mean - mu_ref) / (sd_ref / sqrt(m))."""
    x = np.asarray(proj, dtype=np.float64)
    c = np.vstack([np.zeros((1, x.shape[1])), np.cumsum(x, axis=0)])
    starts = np.arange(n_windows_for(len(x), cfg)) * cfg.stride
    wm = (c[starts + cfg.window] - c[starts]) / cfg.window
    return (wm - ref_prep["mu_marg"]) / (ref_prep["sd_marg"] / math.sqrt(cfg.window))


def cusum_stats_stream(ref_prep: Dict, proj: np.ndarray, cfg: Config) -> np.ndarray:
    """Two-sided per-PC CUSUM on standardized window means, allowance k,
    family-wise max over PCs. No in-chart reset: S+ is computed exactly via
    the prefix-min identity S+_t = P_t - min(0, min_{tau<=t} P_tau); alarm
    clustering is handled by the shared event-dedup cooldown."""
    u = _standardized_window_means(ref_prep, proj, cfg)
    zero = np.zeros((1, u.shape[1]))

    def one_sided(v):
        p = np.cumsum(v - cfg.cusum_k, axis=0)
        run_min = np.minimum.accumulate(np.vstack([zero, p]), axis=0)[1:]
        return p - np.minimum(run_min, 0.0)

    return np.maximum(one_sided(u), one_sided(-u)).max(axis=1)


def ewma_stats_stream(ref_prep: Dict, proj: np.ndarray, cfg: Config) -> np.ndarray:
    """Multivariate EWMA on standardized window means (z_0 = 0): statistic
    ||z_t||^2 * (2 - lambda) / lambda (T^2 with identity covariance — PCs are
    decorrelated under IC; residual miscalibration is absorbed by the
    empirical AIET search)."""
    from scipy.signal import lfilter
    u = _standardized_window_means(ref_prep, proj, cfg)
    lam = cfg.ewma_lambda
    z = lfilter([lam], [1.0, -(1.0 - lam)], u, axis=0)
    return (z ** 2).sum(axis=1) * (2.0 - lam) / lam


def _cusum_reset_scan(u: np.ndarray, cfg: Config, h: float,
                      max_events: Optional[int] = None,
                      chunk: int = 20000) -> Tuple[np.ndarray, int]:
    """Two-sided per-PC CUSUM WITH reset-on-alarm: all charts restart at zero
    after an alarm. Exact segment-wise computation via the prefix-minimum
    identity S+_t = P_t - min(P_{seg-1}, min_{seg<=s<=t} P_s), chunked so the
    total cost is O(n*d) per threshold candidate. Returns (alarm step
    indices, steps scanned) — scanning stops early after `max_events` alarms
    (used by the calibration search)."""
    k = cfg.cusum_k
    p_up = np.cumsum(u - k, axis=0)
    p_dn = np.cumsum(-u - k, axis=0)
    n, d = u.shape
    zeros = np.zeros(d)
    alarms: List[int] = []
    start = 0
    while start < n:
        cur_min_up = (p_up[start - 1] if start > 0 else zeros).copy()
        cur_min_dn = (p_dn[start - 1] if start > 0 else zeros).copy()
        pos, hit = start, None
        while pos < n:
            end = min(pos + chunk, n)
            pu, pd_ = p_up[pos:end], p_dn[pos:end]
            rm_u = np.minimum.accumulate(
                np.vstack([cur_min_up[None], pu]), axis=0)[1:]
            rm_d = np.minimum.accumulate(
                np.vstack([cur_min_dn[None], pd_]), axis=0)[1:]
            stat = np.maximum(pu - rm_u, pd_ - rm_d).max(axis=1)
            over = np.flatnonzero(stat > h)
            if len(over):
                hit = pos + int(over[0])
                break
            cur_min_up, cur_min_dn = rm_u[-1], rm_d[-1]
            pos = end
        if hit is None:
            break
        alarms.append(hit)
        if max_events is not None and len(alarms) >= max_events:
            return np.asarray(alarms, dtype=np.int64), hit + 1
        start = hit + 1
    return np.asarray(alarms, dtype=np.int64), n


def _dedup_events(indices: np.ndarray, cooldown: int) -> List[int]:
    events, nxt = [], -1
    for i in indices:
        if i >= nxt:
            events.append(int(i))
            nxt = i + cooldown
    return events


def calibrate_cusum_reset(u: np.ndarray, cfg: Config):
    """Bisection on the reset-CUSUM chart limit h until the calibration
    estimator (stream-origin-to-first-event plus successive completed gaps
    between cooldown-deduplicated events) lands in the AIET target band.
    The right-censored tail after the final event is not included. The reset
    makes the statistic sequence
    threshold-dependent, so the shared order-statistics search does not
    apply; each candidate h is evaluated by a full scan (with early stop)."""
    target, tol = cfg.target_arl0_steps, cfg.arl0_tol

    def arl_for(h: float):
        al, scanned = _cusum_reset_scan(u, cfg, h, max_events=400)
        ev = _dedup_events(al, cfg.cooldown)
        if not ev:
            return np.inf
        gaps = np.diff(np.concatenate(([-1], ev)))
        return float(gaps.mean())

    # upper bound: the no-reset statistic dominates the reset one
    p = np.cumsum(u - cfg.cusum_k, axis=0)
    hi = float((p - np.minimum(np.minimum.accumulate(p, axis=0), 0.0)).max())
    p = np.cumsum(-u - cfg.cusum_k, axis=0)
    hi = max(hi, float((p - np.minimum(np.minimum.accumulate(p, axis=0),
                                       0.0)).max())) + 1e-9
    lo, best = 0.0, None
    for _ in range(40):
        h = 0.5 * (lo + hi)
        arl = arl_for(h)
        score = (abs(math.log(arl / target))
                 if np.isfinite(arl) and arl > 0 else np.inf)
        if best is None or score < best[0]:
            best = (score, h, arl)
        if target * (1 - tol) <= arl <= target * (1 + tol):
            return h, arl, True
        if arl > target:
            hi = h          # too few alarms -> lower the limit
        else:
            lo = h
    return best[1], best[2], False


def cusum_reset_alarm_bool(u: np.ndarray, cfg: Config, h: float) -> np.ndarray:
    """Boolean alarm sequence of the calibrated reset-CUSUM on a stream."""
    al, _ = _cusum_reset_scan(u, cfg, h)
    out = np.zeros(len(u), dtype=bool)
    out[al] = True
    return out


def compute_window_stats(ref_prep: Dict, wins: np.ndarray, cfg: Config) -> Dict:
    """One statistic per window for the windowed PCA-feature methods (MMD is
    computed stream-wise, see stream_stats; SPE needs the raw feature space
    and has its own path). The histogram counts are computed once and shared
    by KL-Hist / Hellinger-fixed (legacy HDDDM label) / PSI; the Gaussian
    moments once for KL-Gauss / T2 / Frechet. direction='high' alarms on stat
    > threshold; 'low' (KS-fixed, legacy KSWIN min-p label)
    on stat < threshold."""
    m = wins.shape[1]
    counts = window_bin_counts(ref_prep["edges"], wins, cfg.hist_bins)
    q_lap = (counts + 1.0) / (m + cfg.hist_bins)
    q_raw = counts / m
    p_lap = ref_prep["p_lap"][None]
    kl_hist = (q_lap * np.log(q_lap / p_lap)).sum(axis=2).mean(axis=1)
    psi = ((q_lap - p_lap) * np.log(q_lap / p_lap)).sum(axis=2).mean(axis=1)
    bc = np.sqrt(q_raw * ref_prep["p_raw"][None]).sum(axis=2)
    hdddm = np.sqrt(np.clip(1.0 - bc, 0.0, None)).max(axis=1)
    gauss = gauss_moment_stats(ref_prep, wins)
    return {
        "KSWIN": (kswin_minp_stats(ref_prep, wins), "low"),
        "KL-Gauss": (gauss["KL-Gauss"], "high"),
        "KL-Hist": (kl_hist, "high"),
        "HDDDM": (hdddm, "high"),
        "PSI": (psi, "high"),
        "T2": (gauss["T2"], "high"),
        "Frechet": (gauss["Frechet"], "high"),
    }


def stream_stats(ref_prep: Dict, proj: np.ndarray, cfg: Config,
                 chunk: int = 4000) -> Dict:
    """All projection-based method statistics over a whole stream on the
    strided window grid: the windowed methods chunked over windows to bound
    memory, MMD via the exact streaming decomposition, CUSUM/EWMA as
    sequential charts. (SPE is raw-space and handled by its callers.)"""
    n_w = n_windows_for(len(proj), cfg)
    acc: Optional[Dict] = None
    for i0 in range(0, n_w, chunk):
        i1 = min(i0 + chunk, n_w)
        seg = proj[i0 * cfg.stride:(i1 - 1) * cfg.stride + cfg.window]
        wins = make_windows(seg, cfg.window, cfg.stride)
        part = compute_window_stats(ref_prep, wins, cfg)
        if acc is None:
            acc = {m: ([v], d) for m, (v, d) in part.items()}
        else:
            for m, (v, d) in part.items():
                acc[m][0].append(v)
    out = {m: (np.concatenate(vs), d) for m, (vs, d) in acc.items()}
    out["MMD"] = (mmd_stats_stream(ref_prep, proj, cfg), "high")
    out["CUSUM"] = (cusum_stats_stream(ref_prep, proj, cfg), "high")
    out["EWMA"] = (ewma_stats_stream(ref_prep, proj, cfg), "high")
    return out


# ---- sanity checks --------------------------------------------------------
_rng = np.random.default_rng(0)
_ref = _rng.normal(size=(200, 3))
_wins = _rng.normal(size=(5, 30, 3))
_d_mine = ks_d_batched(np.sort(_ref, axis=0), _wins)
for _w in range(5):
    for _j in range(3):
        assert abs(_d_mine[_w, _j]
                   - sstats.ks_2samp(_ref[:, _j], _wins[_w, :, _j]).statistic) < 1e-12

from sklearn.covariance import LedoitWolf as _SkLW
_x = _rng.normal(size=(1, 50, 8)) * np.linspace(0.5, 3.0, 8)
_mu, _cov = ledoit_wolf_batched(_x)
_sk = _SkLW().fit(_x[0])
assert np.allclose(_cov[0], _sk.covariance_, atol=1e-10)
assert np.allclose(_mu[0], _sk.location_, atol=1e-12)

_seq = _rng.normal(size=(200, 2))
assert n_windows_for(200, CFG) == (200 - CFG.window) // CFG.stride + 1
_wv = make_windows(_seq, CFG.window, CFG.stride)
assert np.array_equal(_wv[1], _seq[CFG.stride:CFG.stride + CFG.window])

# streaming MMD must equal the direct per-window estimator exactly
_refp = prepare_reference(_rng.normal(size=(120, 4)), CFG)
_stream = _rng.normal(size=(400, 4))
_direct = mmd_stats(_refp, make_windows(_stream, CFG.window, CFG.stride))
assert np.allclose(mmd_stats_stream(_refp, _stream, CFG), _direct, atol=1e-10)

# vectorized CUSUM/EWMA must match their scalar recursions
_u = _standardized_window_means(_refp, _stream, CFG)
_sp, _ref_c = 0.0, []
for _v in _u[:, 0]:
    _sp = max(0.0, _sp + _v - CFG.cusum_k)
    _ref_c.append(_sp)
_p = np.cumsum(_u[:, 0] - CFG.cusum_k)
_s = _p - np.minimum(np.minimum.accumulate(_p), 0.0)
assert np.allclose(_s, _ref_c, atol=1e-12)
_z, _ref_e = 0.0, []
for _v in _u[:, 0]:
    _z = CFG.ewma_lambda * _v + (1 - CFG.ewma_lambda) * _z
    _ref_e.append(_z)
from scipy.signal import lfilter as _lf
assert np.allclose(_lf([CFG.ewma_lambda], [1.0, -(1.0 - CFG.ewma_lambda)],
                       _u[:, 0]), _ref_e, atol=1e-12)

# reset-on-alarm CUSUM scan must match its scalar recursion exactly
def _slow_reset_cusum(u, k, h):
    sp = np.zeros(u.shape[1])
    sn = np.zeros(u.shape[1])
    out = []
    for i, row in enumerate(u):
        sp = np.maximum(0.0, sp + row - k)
        sn = np.maximum(0.0, sn - row - k)
        if max(sp.max(), sn.max()) > h:
            out.append(i)
            sp[:] = 0.0
            sn[:] = 0.0
    return out

for _h in (2.0, 5.0, 9.0):
    _fast, _ = _cusum_reset_scan(_u, CFG, _h, chunk=37)   # odd chunk on purpose
    assert list(_fast) == _slow_reset_cusum(_u, CFG.cusum_k, _h)

# T2 / Frechet closed forms vs direct per-window computation
from scipy.linalg import sqrtm as _sqrtm
_wt = make_windows(_stream, CFG.window, CFG.stride)[:6]
_gs = gauss_moment_stats(_refp, _wt)
_muw, _covw = ledoit_wolf_batched(np.asarray(_wt, dtype=np.float64))
_covr = _refp["cov_sqrt"] @ _refp["cov_sqrt"]
for _i in range(len(_wt)):
    _d0 = _muw[_i] - _refp["mu"]
    assert abs(_gs["T2"][_i]
               - CFG.window * _d0 @ _refp["cov_inv"] @ _d0) < 1e-8
    _cross = _sqrtm(_refp["cov_sqrt"] @ _covw[_i] @ _refp["cov_sqrt"]).real
    _fd_ref = (_d0 @ _d0 + np.trace(_covw[_i]) + np.trace(_covr)
               - 2.0 * np.trace(_cross))
    assert abs(_gs["Frechet"][_i] - _fd_ref) < 1e-8
# Frechet of the reference against itself must vanish
_gs0 = gauss_moment_stats(_refp, _refp["ref"][None])
assert abs(_gs0["Frechet"][0]) < 1e-8

# SPE norm-difference identity vs explicit residual reconstruction
_pca_t = PCA(n_components=3, random_state=0).fit(_stream)
_pr_t = _pca_t.transform(_stream)
_res_t = _stream - _pca_t.mean_ - _pr_t @ _pca_t.components_
assert np.allclose(pca_spe(_pca_t, _stream, _pr_t),
                   (_res_t ** 2).sum(axis=1), atol=1e-10)

print("sanity checks passed (KS vs scipy; Ledoit-Wolf vs sklearn; windowing; "
      "streaming MMD vs direct; CUSUM/EWMA recursions; reset-CUSUM scan; "
      "T2/Frechet closed forms vs scipy.sqrtm; SPE residual identity)")


In [ ]:
class ConvVAE(nn.Module):
    """Small convolutional VAE for 32x32x3 inputs (latent dim `vae_latent`)."""

    def __init__(self, latent: int = 64):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(inplace=True),     # 16x16
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(inplace=True),    # 8x8
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(inplace=True),   # 4x4
            nn.Flatten())
        self.fc_mu = nn.Linear(128 * 16, latent)
        self.fc_lv = nn.Linear(128 * 16, latent)
        self.fc_dec = nn.Linear(latent, 128 * 16)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid())

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_lv(h)

    def decode(self, z):
        return self.dec(self.fc_dec(z).view(-1, 128, 4, 4))

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        return self.decode(z), mu, lv


def train_seed_vae(seed: int, cfg: Config) -> nn.Module:
    """Per-seed VAE trained on IC-only draws from a dedicated RNG continuation
    (pinned SeedSequence child CHILD_VAE, so the cal/ref/test-stream children
    are untouched). Cached next to the seed's state; the alarm threshold is
    calibrated afterwards by the same empirical-AIET search as every other
    method."""
    outdir = Path(cfg.out_root) / "vae_models"   # extractor-variant independent
    outdir.mkdir(parents=True, exist_ok=True)
    path = outdir / (f"seed{seed}_vae_e{cfg.vae_epochs}_d{cfg.vae_latent}"
                     f"_t{int(round(cfg.ic_train_frac * 100))}"
                     f"_v{int(round(cfg.ic_validation_frac * 100))}.pt")
    if path.exists():
        model = ConvVAE(cfg.vae_latent)
        model.load_state_dict(torch.load(path, map_location="cpu"))
        print(f"[seed {seed}] loaded cached VAE ({path.name})")
        return model.to(DEVICE).eval()

    # With train=50%, the training pool and VAE RNG child are exactly those
    # used by v6. Reuse its checkpoint when available; this reuses fitting,
    # never a threshold or evaluation result.
    baseline_path = (Path(cfg.baseline_root) / "vae_models" /
                     f"seed{seed}_vae_e{cfg.vae_epochs}_d{cfg.vae_latent}_h50.pt")
    if (cfg.reuse_baseline_train_artifacts
            and abs(cfg.ic_train_frac - 0.5) < 1e-12
            and baseline_path.exists()):
        model = ConvVAE(cfg.vae_latent)
        model.load_state_dict(torch.load(baseline_path, map_location="cpu"))
        torch.save(model.state_dict(), path)
        print(f"[seed {seed}] reused training-only v6 VAE ({baseline_path.name})")
        return model.to(DEVICE).eval()

    t0 = time.time()
    child = spawn_children(seed)[CHILD_VAE]
    sampler = EpochCycledSampler(ic_pools(seed, cfg)[0], child)
    torch.manual_seed(seed)
    model = ConvVAE(cfg.vae_latent).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    pool_n = len(sampler.pool_ids)
    for ep in range(cfg.vae_epochs):
        model.train()
        tot = 0.0
        for i in range(0, pool_n, cfg.vae_batch):
            nb = min(cfg.vae_batch, pool_n - i)
            u8, _ = sampler.draw(nb)
            x = to_unit_tensor(u8)
            recon, mu, lv = model(x)
            rec = F.mse_loss(recon, x, reduction="sum") / nb
            kld = -0.5 * torch.sum(1 + lv - mu ** 2 - lv.exp()) / nb
            loss = rec + kld
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.item() * nb
        if ep == 0 or (ep + 1) % 5 == 0 or ep == cfg.vae_epochs - 1:
            print(f"  [seed {seed}] VAE epoch {ep + 1}/{cfg.vae_epochs}: "
                  f"loss {tot / pool_n:.1f}")
    model.eval()
    torch.save(model.state_dict(), path)
    print(f"[seed {seed}] VAE trained ({time.time() - t0:.0f}s)")
    return model


@torch.no_grad()
def vae_recon_errors(model: nn.Module, u8: np.ndarray,
                     batch: int = 1024) -> np.ndarray:
    """Per-sample pixel reconstruction MSE with deterministic encoding
    (decode the posterior mean)."""
    outs = []
    for i in range(0, len(u8), batch):
        x = to_unit_tensor(u8[i:i + batch])
        mu, _ = model.encode(x)
        recon = model.decode(mu)
        outs.append(((recon - x) ** 2).mean(dim=(1, 2, 3)).cpu().numpy())
    return np.concatenate(outs).astype(np.float64)


@torch.no_grad()
def vae_latents(model: nn.Module, u8: np.ndarray, batch: int = 1024) -> np.ndarray:
    """Posterior-mean latent vectors mu(x): the VAE encoder used as a feature
    extractor (block 'vae_latent'), fed to the same PCA + divergence pipeline
    as the ResNet probes."""
    outs = []
    for i in range(0, len(u8), batch):
        mu, _ = model.encode(to_unit_tensor(u8[i:i + batch]))
        outs.append(mu.float().cpu().numpy())
    return np.concatenate(outs)


## Per-seed train/validation calibration protocol (§1a–§1c)

Performed **exactly once per seed**; nothing here is ever touched again by the
16 test streams.

1. **Training partition** — the VAE, blockwise PCA bases, and fixed references
   are fit using only the 3,000 training identities. PCA uses the first 50,000
   epoch-cycled augmented training draws. When compatible v6 training
   artifacts exist they are reused exactly; no v6 thresholds are loaded.
2. **Validation stream** — `cal_arl0_mult × target_arl0` 50-sample
   window-equivalents of IC-only exposure,
   epoch-cycled, augmented draws from the 1,500 validation identities (default
   30×370 = 11,100 window-equivalents = 555,000 samples. With width 50 and
   stride 1 this gives exactly 554,951 overlapping windows/scores. A single
   AIET=370 interval is 18,500 stride steps (samples), and a more stable
   empirical event-interval estimate needs a few dozen expected false alarms per
   candidate threshold). It selects thresholds but fits no representation.
3. **PCA per block** — fit only on the training draws, never on validation or
   test data. The validation stream is projected through the frozen basis.
   Shrinkage-regularized
   (Ledoit–Wolf) covariance is used downstream wherever a covariance is
   estimated, permitting $d=20$ retained PCs at window size 50.
4. **VAE** — trained only on training identities. Its reconstruction errors on
   the validation stream select its alarm threshold by the same procedure as
   every other method.
5. **Fixed reference** — `ref_n` samples from a dedicated training-side RNG
   continuation, projected with the same PCA.
6. **Threshold search per (block, method)** — the detector statistic is
   evaluated on every validation window once; the search is then coarse-to-fine
   (binary search over the order statistics of the calibration stat sequence),
   at each candidate computing completed waits on the stride grid: the first
   event index plus one, followed by successive event-index differences. The
   mean of those completed gaps must land within ±10% of 18,500 steps at the
   default stride; the right-censored tail after the last event is omitted.
   This is a
   search on empirical AIET, not a parametric formula — the same code path
   calibrates the fixed-reference statistics and VAE. KS-fixed uses one
   empirically selected cutoff for its joint minimum-$p$ score; it does not
   apply a separate Bonferroni threshold.
7. Thresholds are cached and reused, **unmodified**, across the drift-free
   verification and all drift streams from the untouched test identities.

**Overlapping windows (stride < 50).** Window starts sit on a grid of `stride`
samples, so consecutive statistics share `window − stride` samples and are
autocorrelated under IC: threshold excursions produce *clusters* of consecutive
alarms. A raw mean gap between threshold crossings would count each cluster many times and bias the
search, so alarms are deduplicated into **events**: after an event, crossings
during the next `ceil(window/stride) − 1` steps are ignored (one full window
of fresh data; cooldown = 1 at
stride 50, i.e. exactly the tumbling behavior). The AIET target is restated on
the step grid as `target_arl0 × window/stride` steps — the same expected
**time** between false-alarm events (370 × 50 samples) at every stride — and
the calibration estimate is converted back by multiplying by
`stride/window`. On the separate verification stream, realised AIET is
instead the full in-control exposure divided by the deduplicated event count.
Detection delays use the same 50-sample window units, so results remain
comparable across strides.


In [ ]:
def alarm_events(alarms: np.ndarray, cooldown: int) -> np.ndarray:
    """Deduplicate a boolean alarm sequence into alarm events: after an event,
    the next `cooldown - 1` steps are ignored (cooldown=1 keeps every alarm,
    which is the tumbling-window case)."""
    idx = np.flatnonzero(alarms)
    events, nxt = [], -1
    for i in idx:
        if i >= nxt:
            events.append(i)
            nxt = i + cooldown
    return np.asarray(events, dtype=np.int64)


def empirical_arl0(scores: np.ndarray, threshold: float, direction: str,
                   cooldown: int = 1):
    """Mean completed event-wait interval in stride steps on an IC score
    sequence: origin-to-first-event plus gaps between later deduplicated
    events. The right-censored tail after the last event is omitted."""
    alarms = scores > threshold if direction == "high" else scores < threshold
    ev = alarm_events(alarms, cooldown)
    if len(ev) == 0:
        return np.inf, 0
    gaps = np.diff(np.concatenate(([-1], ev)))
    return float(gaps.mean()), int(len(ev))


def calibrate_threshold(scores: np.ndarray, direction: str, target: float,
                        tol: float, cooldown: int = 1):
    """Coarse-to-fine (binary) search over the order statistics of the
    calibration score sequence until the completed-gap AIET estimator for
    deduplicated events lands within [target*(1-tol), target*(1+tol)].
    Returns (threshold, achieved_arl0, in_band)."""
    s = np.sort(scores)
    n = len(s)
    if target * (1 - tol) >= n:
        raise ValueError("validation stream too short for the AIET target")

    def threshold_for(k: int) -> float:   # k = intended raw alarm count, 1..n-1
        if direction == "high":
            return 0.5 * (s[n - k - 1] + s[n - k])
        return 0.5 * (s[k - 1] + s[k])

    lo, hi = 1, n - 1
    best = None
    while lo <= hi:
        k = (lo + hi) // 2
        thr = threshold_for(k)
        arl, _ = empirical_arl0(scores, thr, direction, cooldown)
        score = abs(math.log(arl / target)) if np.isfinite(arl) and arl > 0 else np.inf
        if best is None or score < best[0]:
            best = (score, thr, arl)
        if target * (1 - tol) <= arl <= target * (1 + tol):
            return thr, arl, True
        if arl > target:      # too few alarm events at this threshold -> raise k
            lo = k + 1
        else:
            hi = k - 1
    return best[1], best[2], False


def _fit_block_pca(arr: np.ndarray, fit_n: int, cfg: Config) -> PCA:
    """'fixed': pca_dim components. 'evr': fit a capped basis (<=128 PCs),
    then truncate to the smallest d whose cumulative explained variance
    reaches pca_evr (d capped at 128; the dims table in §7 reports what was
    actually retained)."""
    if cfg.pca_criterion == "fixed":
        return PCA(n_components=cfg.pca_dim, svd_solver="randomized",
                   random_state=0).fit(arr[:fit_n])
    cap = int(min(128, arr.shape[1], fit_n))
    p = PCA(n_components=cap, svd_solver="randomized", random_state=0).fit(arr[:fit_n])
    d = int(np.searchsorted(np.cumsum(p.explained_variance_ratio_), cfg.pca_evr)) + 1
    d = min(d, cap)
    p.components_ = p.components_[:d]
    p.explained_variance_ = p.explained_variance_[:d]
    p.explained_variance_ratio_ = p.explained_variance_ratio_[:d]
    p.n_components_ = d
    return p


def run_seed_calibration(seed: int, cfg: Config, extractor: BlockProbeExtractor,
                         variant: str):
    """Fit representations on training identities and select thresholds on
    validation identities for one (extractor variant, seed)."""
    outdir = Path(cfg.out_root) / variant / f"seed{seed}"
    outdir.mkdir(parents=True, exist_ok=True)
    state_path = outdir / "seed_state.pkl"
    if state_path.exists() and not cfg.force_recompute:
        with open(state_path, "rb") as f:
            state = pickle.load(f)
        st_cfg = state.get("config", {})
        if (st_cfg.get("stride", 50) == cfg.stride
                and st_cfg.get("ic_train_frac") == cfg.ic_train_frac
                and st_cfg.get("ic_validation_frac") == cfg.ic_validation_frac
                and state.get("threshold_source") == "validation"
                and ("pixel", "VAE") in state.get("thresholds", {})
                and ("vae_latent", "MMD") in state.get("thresholds", {})
                and ("layer1", "CUSUM") in state.get("thresholds", {})
                and ("layer1", "CUSUM-R") in state.get("thresholds", {})
                and ("layer1", "T2") in state.get("thresholds", {})
                and ("layer1", "SPE") in state.get("thresholds", {})
                and ("layer1", "Frechet") in state.get("thresholds", {})):
            print(f"[seed {seed}] loaded cached seed_state")
            return state
        print(f"[seed {seed}] cached seed_state incompatible "
              f"(stride/split change or missing method blocks) -> recomputing")

    t0 = time.time()
    # VAE first: it consumes its own dedicated RNG continuation, and its recon
    # errors are accumulated during the calibration-stream pass below
    vae = train_seed_vae(seed, cfg)

    ss = np.random.SeedSequence(seed)
    children = spawn_children(seed)
    train_pool, validation_pool, _ = ic_pools(seed, cfg)
    fit_n = cfg.pca_fit_n
    chunk = 5000

    # --- training-only PCA and reference ----------------------------------
    # The default training pool and RNG children are bit-identical to the
    # original v6 fitting side, so compatible representation artifacts can be
    # reused without importing an old threshold or test result.
    pca_models = ref_proj = None
    train_indices = ref_indices = None
    baseline_state_path = (Path(cfg.baseline_root) / variant /
                           f"seed{seed}" / "seed_state.pkl")
    if (cfg.reuse_baseline_train_artifacts
            and abs(cfg.ic_train_frac - 0.5) < 1e-12
            and baseline_state_path.exists()):
        with open(baseline_state_path, "rb") as f:
            old_state = pickle.load(f)
        old_cfg = old_state.get("config", {})
        compatible = (
            old_cfg.get("ic_holdout_frac") == 0.5
            and old_cfg.get("pca_dim") == cfg.pca_dim
            and old_cfg.get("pca_criterion", "fixed") == cfg.pca_criterion
            and old_cfg.get("pca_evr", 0.7) == cfg.pca_evr
            and old_cfg.get("pca_fit_n") == cfg.pca_fit_n
            and old_state.get("extractor_variant") == variant
            and set(FEATURE_BLOCKS) <= set(old_state.get("pca_models", {}))
            and set(FEATURE_BLOCKS) <= set(old_state.get("ref_proj", {})))
        if compatible:
            pca_models = old_state["pca_models"]
            ref_proj = old_state["ref_proj"]
            old_log = old_state.get("train_stream_idx", {})
            old_train = old_log.get("cal_indices", np.empty(0, dtype=np.int64))
            train_indices = np.asarray(old_train)[:fit_n]
            ref_indices = np.asarray(old_log.get(
                "ref_indices", np.empty(0, dtype=np.int64)))
            print(f"[seed {seed}] reused v6 training-only PCA/reference artifacts")

    if pca_models is None:
        train_sampler = EpochCycledSampler(train_pool, children[0])
        raw_buf: Dict[str, list] = {b: [] for b in FEATURE_BLOCKS}
        done = 0
        while done < fit_n:
            nb = min(chunk, fit_n - done)
            u8, _ = train_sampler.draw(nb)
            feats = extractor.extract(u8)
            feats["vae_latent"] = vae_latents(vae, u8)
            for b in FEATURE_BLOCKS:
                raw_buf[b].append(feats[b])
            done += nb
        pca_models = {b: _fit_block_pca(np.concatenate(raw_buf[b]), fit_n, cfg)
                      for b in FEATURE_BLOCKS}
        train_indices = train_sampler.consumed_indices()

        ref_sampler = EpochCycledSampler(train_pool, children[1])
        ref_u8, _ = ref_sampler.draw(cfg.ref_n)
        ref_feats = extractor.extract(ref_u8)
        ref_feats["vae_latent"] = vae_latents(vae, ref_u8)
        ref_proj = {b: pca_models[b].transform(ref_feats[b]).astype(np.float64)
                    for b in FEATURE_BLOCKS}
        ref_indices = ref_sampler.consumed_indices()
        print(f"[seed {seed}] training PCA/reference fitted "
              f"({fit_n:,} + {cfg.ref_n:,} draws)")

    # --- validation-only stream for every threshold -----------------------
    validation_sampler = EpochCycledSampler(
        validation_pool, children[CHILD_VALIDATION])
    proj: Dict[str, list] = {b: [] for b in FEATURE_BLOCKS}
    spe_parts: Dict[str, list] = {b: [] for b in FEATURE_BLOCKS}
    vae_errs: List[np.ndarray] = []
    done = 0
    while done < cfg.cal_samples:
        nb = min(chunk, cfg.cal_samples - done)
        u8, _ = validation_sampler.draw(nb)
        feats = extractor.extract(u8)
        feats["vae_latent"] = vae_latents(vae, u8)
        vae_errs.append(vae_recon_errors(vae, u8))
        for b in FEATURE_BLOCKS:
            pr = pca_models[b].transform(feats[b]).astype(np.float64)
            proj[b].append(pr)
            spe_parts[b].append(pca_spe(pca_models[b], feats[b], pr))
        done += nb
    validation_proj = {b: np.concatenate(proj[b]) for b in FEATURE_BLOCKS}
    validation_spe = {b: np.concatenate(spe_parts[b]) for b in FEATURE_BLOCKS}
    print(f"[seed {seed}] validation stream extracted + projected "
          f"({cfg.cal_samples:,} samples, {time.time() - t0:.0f}s)")

    # --- per-(block, method) empirical-AIET threshold search ----------------
    # search runs on the step grid; achieved AIET stored in window units
    thresholds = {}
    to_windows = cfg.stride / cfg.window

    def _calibrate(key, vals, direction):
        thr, arl, ok = calibrate_threshold(vals, direction, cfg.target_arl0_steps,
                                           cfg.arl0_tol, cfg.cooldown)
        thresholds[key] = {"threshold": float(thr), "direction": direction,
                           "cal_arl0": float(arl * to_windows), "in_band": bool(ok)}
        if not ok:
            print(f"  WARNING seed {seed} {key}: best achievable AIET "
                  f"{arl * to_windows:.0f} windows outside the "
                  f"+/-{cfg.arl0_tol:.0%} band")

    for b in FEATURE_BLOCKS:
        ref_prep = prepare_reference(ref_proj[b], cfg)
        for m, (vals, direction) in stream_stats(
                ref_prep, validation_proj[b], cfg).items():
            _calibrate((b, m), vals, direction)
        # SPE lives in the raw feature space (residual off the PCA basis)
        _calibrate((b, "SPE"), window_means(validation_spe[b], cfg), "high")
        # reset-on-alarm CUSUM: threshold-dependent statistic -> bisection
        u_validation = _standardized_window_means(
            ref_prep, validation_proj[b], cfg)
        thr_r, arl_r, ok_r = calibrate_cusum_reset(u_validation, cfg)
        thresholds[(b, "CUSUM-R")] = {
            "threshold": float(thr_r), "direction": "cusum_reset",
            "cal_arl0": float(arl_r * to_windows), "in_band": bool(ok_r)}
        if not ok_r:
            print(f"  WARNING seed {seed} {(b, 'CUSUM-R')}: best achievable "
                  f"AIET {arl_r * to_windows:.0f} windows outside the "
                  f"+/-{cfg.arl0_tol:.0%} band")
    _calibrate(("pixel", "VAE"), window_means(np.concatenate(vae_errs), cfg), "high")

    state = {
        "seed": seed,
        "extractor_variant": variant,
        "threshold_source": "validation",
        "partition_stream_idx": {
            "pool": f"class{cfg.ic_class}", "seed_entropy": ss.entropy,
            "train_indices": train_indices,
            "validation_indices": validation_sampler.consumed_indices(),
            "ref_indices": ref_indices,
            "partition_sizes": {"train": len(train_pool),
                                "validation": len(validation_pool),
                                "test": len(ic_pools(seed, cfg)[2])}},
        "pca_models": pca_models,
        "ref_proj": ref_proj,
        "vae_ckpt": f"vae_e{cfg.vae_epochs}_d{cfg.vae_latent}.pt",
        "thresholds": thresholds,
        "test_streams": {},
        "config": asdict(cfg),
    }
    with open(state_path, "wb") as f:
        pickle.dump(state, f)
    print(f"[seed {seed}] validation-selected {len(thresholds)} "
          f"(block, method) thresholds ({time.time() - t0:.0f}s total)")
    return state


In [ ]:
SEED_STATES = {}
for variant, extractor in EXTRACTORS.items():
    for seed in variant_seeds(variant, CFG):
        SEED_STATES[(variant, seed)] = run_seed_calibration(
            seed, CFG, extractor, variant)

cal_report = pd.DataFrame([
    {"extractor": v, "seed": s, "block": b, "method": m, **info}
    for (v, s), st in SEED_STATES.items()
    for (b, m), info in st["thresholds"].items()])
cal_report_primary = cal_report[cal_report.extractor == CFG.primary_extractor]
print(f"\nvalidation AIET in window units (target {CFG.target_arl0:.0f}, "
      f"+/-{CFG.arl0_tol:.0%}); {int((~cal_report.in_band).sum())} out-of-band "
      f"across all variants; primary ({CFG.primary_extractor}):")
display(cal_report_primary.pivot_table(index=["block", "method"],
                                       columns="seed", values="cal_arl0").round(1))


## Evaluation metrics

For every (seed × drift-setup × block × method):

- **Realised AIET transfer check** — on the IC-only segment of each test stream (all
  windows ending before the first changepoint sample), the empirical AIET is
  estimated as total IC time / total false-alarm events (deduplicated with the
  same cooldown as calibration), pooled across the 16 streams of a seed and
  reported in window units. It should roughly match the calibrated 370-window
  target — a sanity check that calibration transferred to the test streams.
  With the IC identity split active, this check additionally absorbs the small
  population mismatch between the calibration and held-out halves of the
  class-0 pool, so mild deviation from target is expected there and is itself
  a finding (thresholds tuned on one half of a finite pool, applied to unseen
  images from the other half).
- **Restricted mean detection time (RMDT)** — detection delay counted from the **first observation after the
  changepoint** (the standard sequential change-detection convention): a
  window is a detection opportunity as soon as it contains at least one
  post-change sample, and the delay is the number of post-change samples
  consumed at the end of the first alarming window, in window units — so
  fractional delays below 1 credit alarms raised on partially drifted
  windows (resolution 1/50 window at stride 1). Windows extending past the
  drifted segment's end are not eligible (ambiguous attribution on
  recurring streams). Changepoint-straddling windows therefore count toward
  RMDT but never toward AIET (the IC segment remains windows fully before
  the changepoint), so no window contributes to both metrics. Recurring
  streams contribute one delay per ON-segment (multiple occurrences). A
  missed occurrence is **censored at its segment length** and counted at
  that value (conservative); the detection rate is reported alongside so
  censoring is visible rather than hidden. Raw alarm step indices are
  persisted per (stream, block, method) in `alarm_indices.npz`, so metric
  redefinitions are post-processing rather than re-evaluation.
- Aggregation: mean ± std **across seeds**, per drift setup, per method, per
  block (`pixel` for the VAE). Main tables and rankings use the 16 canonical
  setups; the severity variants are analyzed separately in §5c (their IC
  segments do contribute to the AIET transfer check — in-control time is
  in-control time regardless of what comes after the changepoint).


In [ ]:
def evaluate_seed(seed: int, cfg: Config, extractor: BlockProbeExtractor,
                  state: Dict, variant: str) -> pd.DataFrame:
    """Build the seed's test streams (16 canonical + severity variants) and
    run every (block, method) detector with the seed's frozen PCA/VAE +
    thresholds. Occurrence-level results cached as CSV; stream index logs are
    added to the pickled seed_state."""
    outdir = Path(cfg.out_root) / variant / f"seed{seed}"
    results_path = outdir / "results.csv"
    if results_path.exists() and not cfg.force_recompute:
        df = pd.read_csv(results_path)
        if ("stride" in df.columns and int(df["stride"].iloc[0]) == cfg.stride
                and "ic_train_frac" in df.columns
                and float(df["ic_train_frac"].iloc[0]) == cfg.ic_train_frac
                and "ic_validation_frac" in df.columns
                and float(df["ic_validation_frac"].iloc[0]) == cfg.ic_validation_frac
                and "extractor" in df.columns
                and "severity" in df.columns
                and "delay_rule" in df.columns
                and df["delay_rule"].iloc[0] == "first_obs"
                and (outdir / "alarm_indices.npz").exists()
                and "VAE" in set(df["method"])
                and "vae_latent" in set(df["block"])
                and {"CUSUM", "CUSUM-R", "T2", "SPE", "Frechet"}
                    <= set(df["method"])
                and set(SETUPS) <= set(df["setup"])):
            print(f"[seed {seed}] loaded cached results")
            return df
        print(f"[seed {seed}] cached results incompatible -> recomputing")

    t0 = time.time()
    vae = train_seed_vae(seed, cfg)      # loads the cached checkpoint
    children = spawn_children(seed)      # same pinned registry as calibration
    _, _, ic_test_pool = ic_pools(seed, cfg)
    ooc_pool = np.flatnonzero(LABELS != cfg.ic_class)
    ref_preps = {b: prepare_reference(state["ref_proj"][b], cfg)
                 for b in FEATURE_BLOCKS}
    rows = []
    alarm_log: Dict[str, np.ndarray] = {}   # "<setup>|<block>|<method>" -> step indices
    for si, setup in enumerate(SETUPS):
        subs = children[child_for_setup(si)].spawn(3)
        ic_s = EpochCycledSampler(ic_test_pool, subs[0])
        ooc_s = EpochCycledSampler(ooc_pool, subs[1])
        mix_rng = np.random.default_rng(subs[2])
        imgs, meta = build_test_stream(setup, cfg, ic_s, ooc_s, mix_rng)
        feats = extractor.extract(imgs)
        feats["vae_latent"] = vae_latents(vae, imgs)
        state["test_streams"][setup] = {
            "ic_indices": ic_s.consumed_indices(),
            "ooc_indices": ooc_s.consumed_indices(),
            "schedule": meta["schedule"],
            "changepoints": meta["changepoints"]}

        # window grid in sample units (stride-aware)
        starts = np.arange(n_windows_for(len(imgs), cfg)) * cfg.stride
        ends = starts + cfg.window
        cp_sample = meta["cp"] * cfg.window
        ic_mask = ends <= cp_sample
        ic_time_windows = float(ic_mask.sum() * cfg.stride / cfg.window)

        alarm_sets = []                   # (block, method, boolean alarm array)
        for b in FEATURE_BLOCKS:
            wproj = state["pca_models"][b].transform(feats[b]).astype(np.float64)
            for m, (vals, direction) in stream_stats(ref_preps[b], wproj, cfg).items():
                thr = state["thresholds"][(b, m)]["threshold"]
                alarm_sets.append((b, m, vals > thr if direction == "high"
                                   else vals < thr))
            # SPE: raw-space residual off the PCA basis (Q chart)
            spe_w = window_means(pca_spe(state["pca_models"][b], feats[b],
                                         wproj), cfg)
            alarm_sets.append((b, "SPE",
                               spe_w > state["thresholds"][(b, "SPE")]["threshold"]))
            # reset-CUSUM produces alarms directly (threshold-dependent stat)
            u_str = _standardized_window_means(ref_preps[b], wproj, cfg)
            alarm_sets.append((b, "CUSUM-R", cusum_reset_alarm_bool(
                u_str, cfg, state["thresholds"][(b, "CUSUM-R")]["threshold"])))
        v_vals = window_means(vae_recon_errors(vae, imgs), cfg)
        alarm_sets.append(("pixel", "VAE",
                           v_vals > state["thresholds"][("pixel", "VAE")]["threshold"]))

        for b, m, alarms in alarm_sets:
            alarm_log[f"{setup}|{b}|{m}"] = \
                np.flatnonzero(alarms).astype(np.int32)
            info = state["thresholds"][(b, m)]
            ic_alarms = int(len(alarm_events(alarms & ic_mask, cfg.cooldown)))
            for occ, (st_w, en_w) in enumerate(meta["on_segments"]):
                s0, s1 = st_w * cfg.window, en_w * cfg.window
                # first-observation counting: a window is a detection
                # opportunity as soon as it contains >= 1 post-change sample
                # (ends > s0); delay = post-change samples consumed at the
                # decision, in window units, so fractional values < 1 credit
                # alarms on partially drifted windows. Windows extending past
                # the segment end are excluded (ambiguous attribution on
                # recurring streams).
                elig = np.flatnonzero((ends > s0) & (ends <= s1))
                hit = elig[alarms[elig]]
                detected = hit.size > 0
                delay = (float(ends[hit[0]] - s0) / cfg.window if detected
                         else float(en_w - st_w))
                rows.append({
                    "extractor": variant,
                    "seed": seed, "setup": setup, "kind": meta["kind"],
                    "pattern": meta["pattern"], "transform": meta["transform"],
                    "severity": meta["severity"],
                    "block": b, "method": m, "occurrence": occ,
                    "delay": delay, "detected": bool(detected),
                    "delay_rule": "first_obs",
                    "seg_len": int(en_w - st_w), "stride": cfg.stride,
                    "ic_train_frac": cfg.ic_train_frac,
                    "ic_validation_frac": cfg.ic_validation_frac,
                    "ic_test_frac": 1 - cfg.ic_train_frac - cfg.ic_validation_frac,
                    "ic_windows": ic_time_windows, "ic_alarms": ic_alarms,
                    "cal_arl0": info["cal_arl0"]})
        print(f"  [seed {seed}] {setup} done ({time.time() - t0:.0f}s elapsed)")
    df = pd.DataFrame(rows)
    df.to_csv(results_path, index=False)
    # persist raw alarm step indices so future metric redefinitions are
    # post-processing instead of a re-evaluation (a few kB per seed)
    np.savez_compressed(outdir / "alarm_indices.npz", **alarm_log)
    with open(outdir / "seed_state.pkl", "wb") as f:
        pickle.dump(state, f)
    print(f"[seed {seed}] evaluation finished ({time.time() - t0:.0f}s)")
    return df


RESULTS_ALL = pd.concat(
    [evaluate_seed(seed, CFG, extractor, SEED_STATES[(variant, seed)], variant)
     for variant, extractor in EXTRACTORS.items()
     for seed in variant_seeds(variant, CFG)],
    ignore_index=True)
RESULTS_ALL.to_csv(Path(CFG.out_root, "aggregate", "results_all_seeds.csv"),
                   index=False)
# all main tables/plots below use the primary extractor; the random-control
# comparison has its own section
RESULTS = RESULTS_ALL[RESULTS_ALL.extractor == CFG.primary_extractor].copy()
print(f"\n{len(RESULTS_ALL):,} occurrence-level rows "
      f"({len(RESULTS):,} for the primary extractor)")


## §5a Per-seed realised-AIET verification on untouched test identities

The pre-changepoint segments of the test streams provide only
~150 in-control windows each per seed (~6.5 expected alarm events over the
sixteen canonical streams), enough to
detect gross miscalibration only when pooled across seeds. This section adds
a **dedicated drift-free verification stream per seed**: `null_stream_mult ×
target_arl0` window-equivalents (default 30×370 = the calibration-stream
exposure) drawn
from the **untouched test partition** — unmodified, uncontaminated in-control
images the calibration never saw — via a dedicated RNG child, so nothing
overlaps the calibration or test-stream draw sequences.

With ~30 expected events per (block, method) per seed, each cell supports a
**Poisson-reference diagnostic** of the calibrated operating point: the deduplicated
alarm-event count $N$ is compared against
$N \sim \mathrm{Poisson}(\lambda_0)$, $\lambda_0 = L/\mathrm{ARL}_0^{\;
\mathrm{steps}}$. Two safeguards make the inference honest: (i) with ~61
cells × all seeds examined jointly, reference p-values are Benjamini–Hochberg adjusted
(at nominal calibration, ~5% raw rejections are expected by chance); (ii)
the adequacy of the Poisson reference is checked by a **dispersion index**
(variance/mean of event counts across ten stream segments — identity
recycling could in principle cluster events; dispersion ≈ 1 licenses the
Poisson reference). The resulting intervals and p-values are descriptive
references, not exact inference: overlapping windows, cooldown deduplication,
detector memory, repeated identities, and estimated thresholds violate the
homogeneous independent-Poisson assumptions. Here realised AIET is the full
in-control exposure in 50-sample units divided by the deduplicated event
count; it therefore includes the terminal exposure after the last event. Its
value and ratio to the target are the primary calibration-transfer summaries.


In [ ]:
from scipy.stats import poisson, chi2, false_discovery_control


def _poisson_two_sided(n: int, lam: float) -> float:
    return float(min(1.0, 2.0 * min(poisson.cdf(n, lam),
                                    poisson.sf(n - 1, lam))))


def evaluate_null_stream(seed: int, cfg: Config, extractor: BlockProbeExtractor,
                         state: Dict, variant: str) -> pd.DataFrame:
    """Per-seed drift-free verification stream from the test identities:
    every calibrated detector runs over it, alarm events are counted, and
    each (block, method) cell gets a Poisson-reference count diagnostic of
    the nominal operating point plus a dispersion diagnostic. These are not
    exact tests for the dependent alarm-event process. Cached as CSV."""
    outdir = Path(cfg.out_root) / variant / f"seed{seed}"
    path = outdir / "arl0_null.csv"
    if path.exists() and not cfg.force_recompute:
        df = pd.read_csv(path)
        if ("dispersion" in df.columns
                and {"CUSUM-R", "T2", "SPE", "Frechet"} <= set(df["method"])):
            print(f"[seed {seed}/{variant}] loaded cached null-stream results")
            return df

    t0 = time.time()
    vae = train_seed_vae(seed, cfg)
    child = spawn_children(seed)[CHILD_NULL]
    _, _, ic_test_pool = ic_pools(seed, cfg)
    sampler = EpochCycledSampler(ic_test_pool, child)
    n_samples = int(round(cfg.null_stream_mult * cfg.target_arl0)) * cfg.window
    ref_preps = {b: prepare_reference(state["ref_proj"][b], cfg)
                 for b in FEATURE_BLOCKS}

    proj = {b: [] for b in FEATURE_BLOCKS}
    spe_parts = {b: [] for b in FEATURE_BLOCKS}
    verrs = []
    done, chunk = 0, 5000
    while done < n_samples:
        nb = min(chunk, n_samples - done)
        u8, _ = sampler.draw(nb)
        feats = extractor.extract(u8)
        feats["vae_latent"] = vae_latents(vae, u8)
        verrs.append(vae_recon_errors(vae, u8))
        for b in FEATURE_BLOCKS:
            pr = state["pca_models"][b].transform(feats[b]).astype(np.float64)
            proj[b].append(pr)
            spe_parts[b].append(pca_spe(state["pca_models"][b], feats[b], pr))
        done += nb
    proj = {b: np.concatenate(proj[b]) for b in FEATURE_BLOCKS}
    spe_all = {b: np.concatenate(spe_parts[b]) for b in FEATURE_BLOCKS}
    verrs = np.concatenate(verrs)

    n_steps = n_windows_for(n_samples, cfg)
    lam0 = n_steps / cfg.target_arl0_steps
    time_w = n_steps * cfg.stride / cfg.window
    seg_edges = np.linspace(0, n_steps, 11)
    rows = []

    def add_row(b, m, alarms):
        ev = alarm_events(alarms, cfg.cooldown)
        n_ev = int(len(ev))
        cnt = np.histogram(ev, bins=seg_edges)[0]
        rows.append({
            "extractor": variant, "seed": seed, "block": b, "method": m,
            "events": n_ev, "expected": lam0,
            "arl0_hat": time_w / n_ev if n_ev else np.inf,
            "p_value": _poisson_two_sided(n_ev, lam0),
            "dispersion": (cnt.var(ddof=1) / cnt.mean()
                           if cnt.mean() > 0 else np.nan),
            "stream_windows": time_w})

    for b in FEATURE_BLOCKS:
        for m, (vals, direction) in stream_stats(ref_preps[b], proj[b], cfg).items():
            thr = state["thresholds"][(b, m)]["threshold"]
            add_row(b, m, vals > thr if direction == "high" else vals < thr)
        add_row(b, "SPE", window_means(spe_all[b], cfg)
                > state["thresholds"][(b, "SPE")]["threshold"])
        u = _standardized_window_means(ref_preps[b], proj[b], cfg)
        add_row(b, "CUSUM-R", cusum_reset_alarm_bool(
            u, cfg, state["thresholds"][(b, "CUSUM-R")]["threshold"]))
    add_row("pixel", "VAE",
            window_means(verrs, cfg)
            > state["thresholds"][("pixel", "VAE")]["threshold"])

    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    print(f"[seed {seed}/{variant}] null stream evaluated "
          f"({n_samples:,} held-out draws, {time.time() - t0:.0f}s)")
    return df


if CFG.run_null_stream:
    NULL_RESULTS = pd.concat(
        [evaluate_null_stream(seed, CFG, extractor,
                              SEED_STATES[(variant, seed)], variant)
         for variant, extractor in EXTRACTORS.items()
         for seed in variant_seeds(variant, CFG)],
        ignore_index=True)
    NULL_RESULTS.to_csv(Path(CFG.out_root, "aggregate",
                             "arl0_null_all_seeds.csv"), index=False)

    # --- descriptive Poisson-reference diagnostics + pooled calibration ----
    NP_ = NULL_RESULTS[NULL_RESULTS.extractor == CFG.primary_extractor].copy()
    NP_["p_bh"] = false_discovery_control(NP_["p_value"], method="bh")
    n_rej = int((NP_["p_bh"] < 0.05).sum())
    print(f"\nper-(seed, block, method) Poisson-reference diagnostics: {len(NP_)} cells, "
          f"{n_rej} BH-rejections at 5% "
          f"(raw p<0.05: {int((NP_.p_value < 0.05).sum())}, "
          f"~{0.05 * len(NP_):.0f} expected by chance)")
    print("median dispersion index per method (≈1 licenses the Poisson "
          "reference; unbounded-memory charts are expected to violate it):")
    print(NP_.groupby("method").dispersion.median().round(2).to_string())
    if n_rej:
        print("BH-rejected cells:")
        display(NP_[NP_.p_bh < 0.05][["seed", "block", "method", "events",
                                      "expected", "arl0_hat", "p_bh"]]
                .sort_values("p_bh").head(15).round(3))

    pooled = (NP_.groupby(["block", "method"], as_index=False)
              .agg(events=("events", "sum"), time_w=("stream_windows", "sum")))
    pooled["arl0_hat"] = pooled.time_w / pooled.events.clip(lower=1)
    # Poisson-reference rate interval via the chi-square relation
    pooled["arl0_lo"] = pooled.time_w / (chi2.ppf(0.975,
                                                  2 * (pooled.events + 1)) / 2)
    pooled["arl0_hi"] = pooled.time_w / np.maximum(
        chi2.ppf(0.025, 2 * pooled.events) / 2, 1e-9)
    pooled.to_csv(Path(CFG.out_root, "aggregate", "arl0_null_pooled.csv"),
                  index=False)
    print(f"\npooled realised AIET across seeds (target "
          f"{CFG.target_arl0:.0f}): median {pooled.arl0_hat.median():.0f}, "
          f"cells with CI excluding target: "
          f"{int(((pooled.arl0_hi < CFG.target_arl0) | (pooled.arl0_lo > CFG.target_arl0)).sum())}"
          f" of {len(pooled)}")

    fig, ax = plt.subplots(figsize=(13, 4.5))
    pooled = pooled.sort_values(["method", "block"]).reset_index(drop=True)
    x = np.arange(len(pooled))
    ratio = pooled.arl0_hat / CFG.target_arl0
    yerr = np.vstack([ratio - pooled.arl0_lo / CFG.target_arl0,
                      pooled.arl0_hi / CFG.target_arl0 - ratio])
    ax.errorbar(x, ratio, yerr=np.maximum(yerr, 0), fmt="o", ms=4, capsize=2,
                color="steelblue", ecolor="gray", lw=0.8)
    ax.axhline(1.0, c="crimson", ls="--", lw=1)
    ax.axhspan(0.9, 1.1, color="crimson", alpha=0.08)
    ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{m}\n{b}" for m, b in zip(pooled.method,
                                                    pooled.block)],
                       fontsize=6, rotation=90)
    ax.set_ylabel("realised AIET / target (log)")
    ax.set_title("Per-cell AIET on held-out drift-free streams, pooled "
                 "across seeds (Poisson-reference 95% intervals; band = calibration "
                 "tolerance)")
    fig.tight_layout()
    fig.savefig(Path(CFG.out_root, "aggregate", "arl0_null_check.png"),
                dpi=120)
    plt.show()
else:
    print("run_null_stream = False -> realised-AIET verification stream skipped")


In [ ]:
# --- run-level RMDT (mean delay over drift occurrences within a stream) -----
# main tables/rankings use the 16 CANONICAL setups; the severity variants
# (@s < 1) have their own response section (§5c)
RESULTS_BASE = RESULTS[RESULTS.setup.isin(BASE_SETUPS)]
per_run = (RESULTS_BASE.groupby(["seed", "setup", "kind", "pattern",
                                 "transform", "block", "method"],
                                as_index=False)
           .agg(arl1=("delay", "mean"), det_rate=("detected", "mean")))

# --- per-setup table: rows = method x block, cols = setups, cells = RMDT ----
stats_tbl = (per_run.groupby(["setup", "block", "method"])
             .agg(m=("arl1", "mean"), s=("arl1", "std")).reset_index())
stats_tbl["cell"] = stats_tbl.apply(
    lambda r: f"{r.m:.1f}±{(0.0 if np.isnan(r.s) else r.s):.1f}", axis=1)
per_setup_table = stats_tbl.pivot(index=["method", "block"], columns="setup",
                                  values="cell")[BASE_SETUPS]
per_setup_table.to_csv(Path(CFG.out_root, "aggregate", "arl1_per_setup.csv"))
print("RMDT (windows), mean±std across seeds:")
display(per_setup_table)

# --- realised-AIET transfer check on test-stream IC segments ----------------
# pooled over ALL streams incl. severity variants: their pre-changepoint
# segments are equally valid in-control time
ic = RESULTS.drop_duplicates(["seed", "setup", "block", "method"])
arl0_check = (ic.groupby(["seed", "block", "method"])
              .agg(icw=("ic_windows", "sum"), ica=("ic_alarms", "sum"))
              .reset_index())
arl0_check["arl0_hat"] = np.where(arl0_check.ica > 0,
                                  arl0_check.icw / np.maximum(arl0_check.ica, 1),
                                  np.inf)
arl0_table = (arl0_check.groupby(["block", "method"])
              .agg(test_arl0=("arl0_hat", "mean")).reset_index()
              .merge(cal_report_primary.groupby(["block", "method"])
                     .agg(cal_arl0=("cal_arl0", "mean")).reset_index(),
                     on=["block", "method"]))
arl0_table["target"] = CFG.target_arl0
arl0_table.to_csv(Path(CFG.out_root, "aggregate", "arl0_check.csv"), index=False)
print("\nRealised-AIET transfer check in window units (test-stream IC segments vs "
      "calibration; inf = zero false alarms observed):")
display(arl0_table.round(1))

# --- holistic ranking at a common nominal validation-AIET target ------------
holistic = (per_run.groupby(["method", "block"])
            .agg(mean_arl1=("arl1", "mean"), mean_det_rate=("det_rate", "mean"))
            .sort_values("mean_arl1"))
holistic.to_csv(Path(CFG.out_root, "aggregate", "holistic_ranking.csv"))
print("\nHolistic ranking (RMDT across all 16 setups; common nominal validation AIET,"
      " realised test AIETs may differ):")
display(holistic.round(2))
print("\nMethod-level (feature blocks pooled):")
display(per_run.groupby("method")
        .agg(mean_arl1=("arl1", "mean"), mean_det_rate=("det_rate", "mean"))
        .sort_values("mean_arl1").round(2))

# --- KL-Gauss vs KL-Hist agreement (report both if they diverge) ------------
_klg = per_run[per_run.method == "KL-Gauss"].set_index(["seed", "setup", "block"]).arl1
_klh = per_run[per_run.method == "KL-Hist"].set_index(["seed", "setup", "block"]).arl1
print(f"\nKL-Gauss vs KL-Hist RMDT: Spearman rho = "
      f"{_klg.corr(_klh, method='spearman'):.3f}, "
      f"mean |RMDT difference| = {(_klg - _klh).abs().mean():.1f} windows")


In [ ]:
from matplotlib.lines import Line2D

# --- RMDT by method, per temporal drift pattern -----------------------------
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), sharey=True)
for ax, pat in zip(axes, PATTERNS):
    sub = per_run[per_run.pattern == pat]
    ax.boxplot([sub[sub.method == m].arl1.values for m in ALL_METHODS],
               tick_labels=ALL_METHODS)
    ax.set_title(pat)
    ax.tick_params(axis="x", rotation=60)
axes[0].set_ylabel("RMDT (windows)")
fig.suptitle("Detection delay by method per temporal pattern "
             "(all setups, blocks, seeds; censored at segment length)")
fig.tight_layout()
fig.savefig(Path(CFG.out_root, "aggregate", "arl1_by_pattern.png"), dpi=120)
plt.show()

# --- bar charts: RMDT per detection method, one panel per drift setup --------
# method bars average over the ResNet blocks only; the two VAE-based
# detectors get explicit, separately labeled bars (VAE-recon = pixel
# reconstruction error; VAE-latent = the divergence/chart methods applied to
# the VAE latent block, averaged over methods)
colors = dict(zip(ALL_METHODS, plt.cm.tab20.colors))   # 13 methods > tab10
colors["VAE-recon"] = colors["VAE"]
colors["VAE-latent"] = "teal"
BAR_METHODS = [m for m in ALL_METHODS if m != "VAE"] + ["VAE-recon", "VAE-latent"]
_bar_parts = pd.concat([
    (per_run[per_run.block.isin(BLOCKS)]
     .groupby(["seed", "setup", "method"], as_index=False).arl1.mean()),
    (per_run[per_run.block == "vae_latent"]
     .groupby(["seed", "setup"], as_index=False).arl1.mean()
     .assign(method="VAE-latent")),
    (per_run[per_run.block == "pixel"]
     .groupby(["seed", "setup"], as_index=False).arl1.mean()
     .assign(method="VAE-recon")),
], ignore_index=True)
bar_src = (_bar_parts.groupby(["setup", "method"]).arl1
           .agg(["mean", "std"]).reset_index())
fig, axes = plt.subplots(4, 4, figsize=(19, 13))
for ax, setup in zip(axes.ravel(), BASE_SETUPS):
    sub = (bar_src[bar_src.setup == setup].set_index("method")
           .reindex(BAR_METHODS))
    ax.bar(range(len(BAR_METHODS)), sub["mean"], yerr=sub["std"].fillna(0.0),
           color=[colors[m] for m in BAR_METHODS], capsize=3,
           edgecolor="k", linewidth=0.4)
    ax.set_xticks(range(len(BAR_METHODS)))
    ax.set_xticklabels(BAR_METHODS, rotation=60, fontsize=7)
    ax.set_title(setup, fontsize=9)
for r in range(4):
    axes[r, 0].set_ylabel("RMDT (windows)")
fig.suptitle("RMDT per detection method for each drift setup "
             "(method bars = ResNet blocks averaged; VAE-latent = methods on "
             "the latent block, averaged; whiskers = std across seeds)")
fig.tight_layout()
fig.savefig(Path(CFG.out_root, "aggregate", "arl1_bars_per_setup.png"), dpi=120)
plt.show()

# --- realised AIET vs RMDT (nominally validation-matched operating points) ---
sc = (per_run.groupby(["method", "block"]).agg(mean_arl1=("arl1", "mean"))
      .reset_index()
      .merge(arl0_check.groupby(["block", "method"])
             .agg(test_arl0=("arl0_hat", "mean")).reset_index(),
             on=["method", "block"]))
_finite = sc.test_arl0[np.isfinite(sc.test_arl0)]
_cap = 2 * _finite.max() if len(_finite) else 2 * CFG.target_arl0
markers = dict(zip(FEATURE_BLOCKS + ("pixel",), "osD^XP"))
colors = dict(zip(ALL_METHODS, plt.cm.tab20.colors))
fig, ax = plt.subplots(figsize=(7.5, 5))
for _, r in sc.iterrows():
    ax.scatter(min(r.test_arl0, _cap), r.mean_arl1,
               marker=markers[r.block], color=colors[r.method], s=70,
               edgecolor="k", linewidth=0.4)
ax.axvline(CFG.target_arl0, ls="--", c="gray", label=f"AIET target {CFG.target_arl0:.0f}")
ax.set_xlabel("realised AIET on test-stream IC segments (windows, capped)")
ax.set_ylabel("RMDT (windows)")
ax.set_title("Realised AIET vs RMDT across all setups")
handles = ([Line2D([], [], marker="o", ls="", color=colors[m], label=m)
            for m in ALL_METHODS]
           + [Line2D([], [], marker=markers[b], ls="", color="gray", label=b)
              for b in FEATURE_BLOCKS + ("pixel",)])
ax.legend(handles=handles, fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(Path(CFG.out_root, "aggregate", "arl0_vs_arl1.png"), dpi=120)
plt.show()

# The former Friedman--Nemenyi critical-difference analysis is intentionally
# omitted. The manuscript's direct comparisons use paired seed-level summaries,
# two-sided Wilcoxon signed-rank tests, Holm adjustment across the 55 retained-
# method contrasts, and paired seed bootstrap intervals. Seed--setup rows are
# not treated as independent inferential replicates.

# --- Kaplan-Meier detection-delay survival curves ---------------------------
# Censoring-honest alternative to the censored-at-segment-length means: a
# missed occurrence exits the risk set at its segment length instead of being
# counted as a fake delay value.
def km_curve(delays, detected):
    order = np.argsort(delays)
    d = np.asarray(delays, dtype=float)[order]
    e = np.asarray(detected, dtype=bool)[order]
    at_risk, s = len(d), 1.0
    ts, ss = [0.0], [1.0]
    for t in np.unique(d):
        m = d == t
        deaths = int((e & m).sum())
        if deaths and at_risk:
            s *= 1.0 - deaths / at_risk
            ts.append(float(t))
            ss.append(s)
        at_risk -= int(m.sum())
    return np.array(ts), np.array(ss)


fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), sharey=True)
for ax, pat in zip(axes, PATTERNS):
    sub = RESULTS_BASE[RESULTS_BASE.pattern == pat]
    for m in ALL_METHODS:
        sm = sub[sub.method == m]
        if len(sm):
            ts, ss = km_curve(sm.delay.values, sm.detected.values)
            ax.step(ts, ss, where="post", color=colors[m], label=m, lw=1.4)
    ax.set_title(pat)
    ax.set_xlabel("delay (windows)")
axes[0].set_ylabel("P(not yet detected)")
axes[0].legend(fontsize=7)
fig.suptitle("Kaplan-Meier detection-delay curves per temporal pattern "
             "(all setups, blocks, seeds; censored occurrences handled correctly)")
fig.tight_layout()
fig.savefig(Path(CFG.out_root, "aggregate", "arl1_survival.png"), dpi=120)
plt.show()

print(f"\nAll tables and figures saved under {CFG.out_root}/aggregate/")


## §5c Severity response: detection vs drift magnitude

Every canonical concept/label stream drifts to **full severity** ($s=1$), so
the main tables measure delay at one drift magnitude only — and detector
rankings are not magnitude-invariant: a method that wins at a blatant shift
can be useless at a subtle one (drift-*magnitude* is a first-class descriptor
of a drift, alongside its temporal pattern). This section reads out the
**severity–response curve** on the sudden setups: the same changepoint
geometry at $s \in$ `severity_levels`, where $s$ scales the transform-severity
map (concept) or the OOC-injection rate (label). Sudden-only by design: under
a ramp, per-window severity is already confounded with time since onset.

Reported per method (ResNet blocks averaged; VAE-recon and VAE-latent
separate, as in the bar charts): RMDT and detection rate per severity
level. The interesting read-outs are (i) the **detection floor** — the
severity below which a method stops detecting within the horizon — and (ii)
**rank crossings** between severity levels, which quantify how far the s=1
conclusions generalize toward subtler drift.


In [ ]:
# --- severity response on the sudden setups ---------------------------------
sev_src = RESULTS[RESULTS.pattern == "sudden"]
_sev_parts = pd.concat([
    (sev_src[sev_src.block.isin(BLOCKS)]
     .groupby(["seed", "kind", "severity", "method"], as_index=False)
     .agg(arl1=("delay", "mean"), det=("detected", "mean"))),
    (sev_src[sev_src.block == "vae_latent"]
     .groupby(["seed", "kind", "severity"], as_index=False)
     .agg(arl1=("delay", "mean"), det=("detected", "mean"))
     .assign(method="VAE-latent")),
    (sev_src[sev_src.block == "pixel"]
     .groupby(["seed", "kind", "severity"], as_index=False)
     .agg(arl1=("delay", "mean"), det=("detected", "mean"))
     .assign(method="VAE-recon")),
], ignore_index=True)
sev_tbl = (_sev_parts.groupby(["kind", "method", "severity"], as_index=False)
           .agg(arl1=("arl1", "mean"), det=("det", "mean")))
sev_tbl.to_csv(Path(CFG.out_root, "aggregate", "severity_response.csv"),
               index=False)
print("RMDT (windows) on sudden setups by drift severity "
      "(concept: transforms averaged; label: OOC rate = s * ooc_rate_max):")
display(sev_tbl.pivot_table(index="method", columns=["kind", "severity"],
                            values="arl1").round(1))
print("\ndetection rate:")
display(sev_tbl.pivot_table(index="method", columns=["kind", "severity"],
                            values="det").round(2))

_sev_colors = dict(zip(ALL_METHODS, plt.cm.tab20.colors))
_sev_colors["VAE-recon"] = _sev_colors["VAE"]
_sev_colors["VAE-latent"] = "teal"
_sev_methods = [m for m in ALL_METHODS if m != "VAE"] + ["VAE-recon",
                                                         "VAE-latent"]
sev_levels = sorted(sev_tbl.severity.unique())
fig, axes = plt.subplots(2, 2, figsize=(12.5, 8), sharex=True)
for ci, kind in enumerate(("concept", "label")):
    sub = sev_tbl[sev_tbl.kind == kind]
    for m in _sev_methods:
        sm = sub[sub.method == m].set_index("severity").reindex(sev_levels)
        axes[0, ci].plot(sev_levels, sm.arl1, marker="o", lw=1.3,
                         color=_sev_colors[m], label=m)
        axes[1, ci].plot(sev_levels, sm.det, marker="o", lw=1.3,
                         color=_sev_colors[m], label=m)
    axes[0, ci].set_title(f"{kind} drift (sudden)")
    axes[1, ci].set_xlabel("drift severity s")
    axes[1, ci].set_xticks(sev_levels)
    axes[1, ci].set_ylim(-0.03, 1.03)
axes[0, 0].set_ylabel("RMDT (windows)")
axes[1, 0].set_ylabel("detection rate")
axes[0, 1].legend(fontsize=7, ncol=2)
fig.suptitle("Severity response at the nominal validation-AIET target (sudden setups; "
             "missed occurrences censored at segment length)")
fig.tight_layout()
fig.savefig(Path(CFG.out_root, "aggregate", "severity_response.png"), dpi=120)
plt.show()


## §5b Frozen-extractor comparison: ImageNet-pretrained vs random-init

Both extractors are frozen and CIFAR-naive; they differ only in whether the
weights encode anything *learned*. If the random network (a structured random
nonlinear projection) detects a drift family as well as the ImageNet one,
detection there relies on architecture + calibration, not on learned
representations — and, since the random weights have never seen any data, it
also rules out any extractor-side exposure as an explanation. The prediction:
the gap should be small for low-level image transformations (JPEG/stretch/saturation)
and largest for OOC contamination, whose signal lives along semantic directions that
only training creates. Compared on the shared seeds, ResNet blocks only (the
VAE-based rows are extractor-independent by construction).


In [ ]:
if CFG.compare_random_extractor:
    shared_seeds = [s for s in CFG.random_extractor_seeds if s in CFG.seeds]
    cmp = RESULTS_ALL[RESULTS_ALL.seed.isin(shared_seeds)
                      & RESULTS_ALL.block.isin(BLOCKS)
                      & RESULTS_ALL.setup.isin(BASE_SETUPS)]
    cr = (cmp.groupby(["extractor", "seed", "setup", "kind", "block", "method"],
                      as_index=False)
          .agg(arl1=("delay", "mean"), det=("detected", "mean")))
    cr.to_csv(Path(CFG.out_root, "aggregate", "extractor_comparison.csv"),
              index=False)

    print(f"RMDT (windows) on shared seeds {shared_seeds}, ResNet blocks only:")
    tbl = cr.pivot_table(index="method", columns=["kind", "extractor"],
                         values="arl1")
    display(tbl.round(1))
    print("\nby block:")
    display(cr.pivot_table(index="block", columns=["kind", "extractor"],
                           values="arl1").round(1))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
    width = 0.38
    for ax, kind in zip(axes, ("concept", "label")):
        sub = cr[cr.kind == kind]
        means = sub.pivot_table(index="method", columns="extractor",
                                values="arl1").reindex(list(METHODS))
        x = np.arange(len(means))
        ax.bar(x - width / 2, means[CFG.primary_extractor], width,
               label=CFG.primary_extractor, color="steelblue",
               edgecolor="k", linewidth=0.4)
        ax.bar(x + width / 2, means["random"], width, label="random",
               color="darkorange", edgecolor="k", linewidth=0.4)
        ax.set_xticks(x)
        ax.set_xticklabels(means.index, rotation=60, fontsize=8)
        ax.set_title(f"{kind} drift")
    axes[0].set_ylabel("RMDT (windows)")
    axes[0].legend()
    fig.suptitle("Frozen ImageNet-pretrained vs frozen random-init extractor "
                 "(shared seeds, ResNet blocks, common nominal validation AIET)")
    fig.tight_layout()
    fig.savefig(Path(CFG.out_root, "aggregate", "extractor_comparison.png"),
                dpi=120)
    plt.show()
else:
    print("compare_random_extractor = False -> comparison skipped")


## Optional development module: alarm-triggered updating (disabled)

**This module is not executed in the reported benchmark, and none of its outputs are used in the thesis.** It is retained only as development code for a future post-alarm study with its own reference and calibration design.

Once a detector fires, the monitoring system must *react*. This section
implements and compares three retraining policies for the detector's **model
of normality** (reference sample + threshold — the retrainable part of this
pipeline; the shared extractor stays frozen per §0, and fine-tuning the
VAE/classifier is the natural extension discussed in the text):

- **static** — never retrain (the benchmark's default behavior). After real
  drift this produces perpetual alarming: the post-changepoint alarm-event
  rate stays orders of magnitude above the nominal 1/370 per window (alarm
  fatigue).
- **triggered** — retrain on *confirmed detection*: `adapt_confirm_events`
  alarm events within `adapt_confirm_horizon` windows confirm a regime change
  (at nominal AIET=370 the probability of spuriously clustering that many false-alarm
  events is negligible, protecting the retraining budget). The next
  `adapt_buffer_windows` windows are collected as an adaptation buffer; its
  first half becomes the new reference, the second half sets a **plug-in
  quantile threshold** at the nominal per-step alarm rate. Retraining cost:
  exactly one adaptation per drift episode, spent only when needed.
- **periodic-T** — retrain every `T` windows from the most recent buffer,
  drift or no drift, for several `T`. This is the "optimal time interval"
  strawman: choosing `T` well requires knowing the drift spacing in advance —
  too small wastes retrains on in-control data (and each retrain risks
  absorbing an active drift silently), too large leaves the model stale for
  up to `T` windows after each change.

Metrics per (seed × stream × policy): number of adaptations (compute/labeling
cost), detection-to-adaptation delay, the **post-adaptation alarm-event rate**
(after adapting, the drifted regime is the new normal, so this should return
to the nominal 1/370 per window — the quantitative test of whether adaptation
absorbed the regime), and for recurring streams, whether **subsequent regime
flips are still re-detected** after adapting to the first ON regime.

Honest caveat: the plug-in threshold is calibrated on a short buffer
(~`adapt_buffer_windows/2` windows) instead of the full §1c empirical-AIET
search, so its realized false-alarm rate is noisier than the calibrated 1/370
— the gap between the triggered policy's realized rate and nominal *is the
measured price of fast adaptation*, and a full §1c recalibration on
subsequently accumulated data is the documented follow-up step.


In [ ]:
if CFG.run_adaptation:
    MON_BLOCK, MON_METHOD = CFG.adapt_monitor
    _EX = EXTRACTORS[CFG.primary_extractor]
    assert MON_METHOD not in ("CUSUM-R", "SPE", "VAE"), \
        "the retraining monitor must be a projection-based threshold-on-statistic method"
    assert (MON_BLOCK, MON_METHOD) in SEED_STATES[(CFG.primary_extractor,
                                                   CFG.seeds[0])]["thresholds"]


    def confirm_time(events: np.ndarray, need: int, horizon_steps: int):
        """First step index at which `need` alarm events have occurred within
        `horizon_steps` of each other (drift confirmation rule)."""
        for i in range(len(events) - need + 1):
            if events[i + need - 1] - events[i] <= horizon_steps:
                return int(events[i + need - 1])
        return None


    def _mon_stats(ref_prep, proj, cfg):
        vals, direction = stream_stats(ref_prep, proj, cfg)[MON_METHOD]
        return vals, direction


    def _retrain_from_buffer(buf, cfg):
        """Retrain the normality model from an adaptation buffer: first half ->
        new reference, second half -> plug-in quantile threshold at the nominal
        per-step alarm rate (see markdown caveat)."""
        half = len(buf) // 2
        ref_prep = prepare_reference(buf[:half], cfg)
        vals, direction = _mon_stats(ref_prep, buf[half:], cfg)
        q = 1.0 - 1.0 / cfg.target_arl0_steps
        thr = (float(np.quantile(vals, q)) if direction == "high"
               else float(np.quantile(vals, 1.0 - q)))
        return ref_prep, thr, direction


    def _event_rate_per_window(vals, thr, direction, cfg):
        alarms = vals > thr if direction == "high" else vals < thr
        ev = alarm_events(alarms, cfg.cooldown)
        time_w = len(vals) * cfg.stride / cfg.window
        return len(ev) / max(time_w, 1e-9), ev


    t0 = time.time()
    adapt_rows = []
    buf_n = CFG.adapt_buffer_windows * CFG.window
    horizon_steps = int(CFG.adapt_confirm_horizon * CFG.window / CFG.stride)

    for seed in CFG.seeds:
        state = SEED_STATES[(CFG.primary_extractor, seed)]
        vae = train_seed_vae(seed, CFG) if MON_BLOCK == "vae_latent" else None
        info0 = state["thresholds"][(MON_BLOCK, MON_METHOD)]
        ref_prep0 = prepare_reference(state["ref_proj"][MON_BLOCK], CFG)
        children = spawn_children(seed)
        _, _, ic_test_pool = ic_pools(seed, CFG)
        ooc_pool = np.flatnonzero(LABELS != CFG.ic_class)
        # canonical setups only: severity variants add nothing to the policy
        # comparison and would inflate the runtime of this section by 50%
        for si, setup in enumerate(BASE_SETUPS):
            subs = children[child_for_setup(si)].spawn(3)
            imgs, meta = build_test_stream(
                setup, CFG, EpochCycledSampler(ic_test_pool, subs[0]),
                EpochCycledSampler(ooc_pool, subs[1]), np.random.default_rng(subs[2]))
            if MON_BLOCK == "vae_latent":
                raw = vae_latents(vae, imgs)
            else:
                raw = _EX.extract(imgs)[MON_BLOCK]
            proj = state["pca_models"][MON_BLOCK].transform(raw).astype(np.float64)
            vals0, dir0 = _mon_stats(ref_prep0, proj, CFG)
            alarms0 = (vals0 > info0["threshold"] if dir0 == "high"
                       else vals0 < info0["threshold"])
            events0 = alarm_events(alarms0, CFG.cooldown)
            starts = np.arange(len(vals0)) * CFG.stride
            cp_sample = meta["cp"] * CFG.window
            s_sched = meta["schedule"]
            flips = [w * CFG.window for w in range(1, len(s_sched))
                     if s_sched[w] != s_sched[w - 1]]
            base = dict(seed=seed, setup=setup, kind=meta["kind"],
                        pattern=meta["pattern"])

            # ---- static: never retrain ----------------------------------------
            post_time = (starts >= cp_sample).sum() * CFG.stride / CFG.window
            ev_post = sum(1 for e in events0 if starts[e] >= cp_sample)
            adapt_rows.append({**base, "policy": "static", "n_adapts": 0,
                               "adapt_delay_w": np.nan,
                               "post_event_rate": ev_post / max(post_time, 1e-9),
                               "flips_redetected": np.nan})

            # ---- triggered: retrain on confirmed detection ---------------------
            row = {**base, "policy": "triggered", "n_adapts": 0,
                   "adapt_delay_w": np.nan, "post_event_rate": np.nan,
                   "flips_redetected": np.nan}
            t_c = confirm_time(events0, CFG.adapt_confirm_events, horizon_steps)
            if t_c is not None:
                buf_s = int(starts[t_c]) + CFG.window
                buf_e = buf_s + buf_n
                if len(proj) - buf_e >= 2 * CFG.window:
                    ref1, thr1, dir1 = _retrain_from_buffer(proj[buf_s:buf_e], CFG)
                    vals1, _ = _mon_stats(ref1, proj[buf_e:], CFG)
                    rate1, ev1 = _event_rate_per_window(vals1, thr1, dir1, CFG)
                    row.update(n_adapts=1,
                               adapt_delay_w=(buf_e - cp_sample) / CFG.window,
                               post_event_rate=rate1)
                    later = [f for f in flips if f > buf_e]
                    if later:
                        ev1_samples = buf_e + ev1 * CFG.stride
                        hits = 0
                        for f, fe in zip(later, later[1:] + [len(proj)]):
                            if np.any((ev1_samples >= f) & (ev1_samples < fe)):
                                hits += 1
                        row["flips_redetected"] = hits / len(later)
            adapt_rows.append(row)

            # ---- periodic-T: retrain on a clock --------------------------------
            for T in CFG.adapt_periodic_T:
                t_s = T * CFG.window
                n_adapts, ev_count, time_post = 0, 0, 0.0
                t = t_s
                while t + CFG.window <= len(proj):
                    buf = proj[max(0, t - buf_n):t]
                    if len(buf) >= 4 * CFG.window:
                        refT, thrT, dirT = _retrain_from_buffer(buf, CFG)
                        n_adapts += 1
                        seg = proj[t:min(t + t_s, len(proj))]
                        if len(seg) >= CFG.window:
                            valsT, _ = _mon_stats(refT, seg, CFG)
                            _, evT = _event_rate_per_window(valsT, thrT, dirT, CFG)
                            seg_starts = t + np.arange(len(valsT)) * CFG.stride
                            ev_count += sum(1 for e in evT
                                            if seg_starts[e] >= cp_sample)
                            time_post += ((seg_starts >= cp_sample).sum()
                                          * CFG.stride / CFG.window)
                    t += t_s
                adapt_rows.append({**base, "policy": f"periodic-{T}",
                                   "n_adapts": n_adapts, "adapt_delay_w": np.nan,
                                   "post_event_rate": ev_count / max(time_post, 1e-9),
                                   "flips_redetected": np.nan})
        print(f"[seed {seed}] retraining-policy experiment done "
              f"({time.time() - t0:.0f}s elapsed)")

    ADAPT = pd.DataFrame(adapt_rows)
    ADAPT.to_csv(Path(CFG.out_root, "aggregate", "adaptation_policies.csv"), index=False)

    print(f"\nmonitor = {MON_METHOD} on {MON_BLOCK}; nominal IC event rate "
          f"= {1 / CFG.target_arl0:.4f} events/window")
    summary = (ADAPT.groupby("policy")
               .agg(adapts_per_stream=("n_adapts", "mean"),
                    detect_to_adapt_delay_w=("adapt_delay_w", "mean"),
                    post_adapt_event_rate=("post_event_rate", "mean"),
                    streams_adapted=("n_adapts", lambda s: (s > 0).mean())))
    display(summary.round(4))
    _fl = ADAPT[(ADAPT.policy == "triggered") & ADAPT.flips_redetected.notna()]
    if len(_fl):
        print(f"recurring streams: fraction of post-adaptation regime flips "
              f"re-detected = {_fl.flips_redetected.mean():.2f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    pols = summary.index.tolist()
    axes[0].bar(pols, summary.post_adapt_event_rate, color="steelblue",
                edgecolor="k", linewidth=0.4)
    axes[0].axhline(1 / CFG.target_arl0, ls="--", c="crimson",
                    label=f"nominal 1/{CFG.target_arl0:.0f}")
    axes[0].set_yscale("log")
    axes[0].set_ylabel("post-drift alarm-event rate (events/window)")
    axes[0].legend(fontsize=8)
    axes[1].bar(pols, summary.adapts_per_stream, color="darkorange",
                edgecolor="k", linewidth=0.4)
    axes[1].set_ylabel("retrainings per stream (cost)")
    for ax in axes:
        ax.tick_params(axis="x", rotation=30)
    fig.suptitle("Retraining policies: regime absorption vs retraining cost "
                 f"({MON_METHOD} on {MON_BLOCK})")
    fig.tight_layout()
    fig.savefig(Path(CFG.out_root, "aggregate", "adaptation_policies.png"), dpi=120)
    plt.show()

else:
    print('run_adaptation = False -> ancillary adaptation experiment skipped')


## Optional development module: fixed-d PCA sweep (disabled)

**This module is not executed in the reported benchmark, and none of its outputs are used in the thesis.** The reported benchmark retains a **fixed d = 20 PCs in every block**, so the
dimension and aggregation burden (KS-fixed), covariance-estimation burden (KL-Gauss), and
averaging dilution (KL-Hist/PSI) are identical across blocks and differences
are attributable to the representations. To test whether the fixed count is
adequate rather than replace it with an explained-variance rule (which would
give unequal per-block d and reintroduce that confound, and push the deep
blocks past the window size where covariance estimation fails), this section
runs a **uniform fixed-d sweep**: the same d is applied to every block, for
d in `ablation_dims`, all below the window size. The main run's `pca_dim`
(e.g. 20) is the sweep's middle point and is reused directly from `RESULTS`,
not recomputed; only the extra dims run, on `ablation_seeds`.

What the sweep shows: whether the method ranking is stable across d. More PCs
expose more drift-carrying directions but (a) change the distribution of
KS-fixed's empirically calibrated minimum-$p$ score, (b)
strain window-fitted covariances (KL-Gauss) as d approaches the window size,
and (c) water down mean-aggregated statistics with noise dimensions. Because
every setting uses one d for all blocks, dimension parity is preserved and
the comparison isolates the count. The per-seed VAE is reused from the main
run (PCA-independent); thresholds are fully recalibrated per setting, so
every setting targets the same nominal validation AIET.


In [ ]:
from dataclasses import replace
import shutil

if CFG.run_pca_ablation:
    # sweep the extra dims only; the main run (pca_dim) is reused from RESULTS.
    # each arm writes to its own out_root so runs_v6 is never touched, and the
    # frozen extractor + per-seed VAE checkpoints are reused (no retraining).
    sweep_dims = sorted(set(CFG.ablation_dims) - {CFG.pca_dim})
    abl_frames = []
    for d in sweep_dims:
        label = f"d={d}"
        cfg_a = replace(CFG, pca_criterion="fixed", pca_dim=int(d),
                        out_root=str(Path(CFG.out_root) / f"ablation_k{d}"))
        Path(cfg_a.out_root, "aggregate").mkdir(parents=True, exist_ok=True)
        for seed in CFG.ablation_seeds:
            vae_name = (f"seed{seed}_vae_e{CFG.vae_epochs}_d{CFG.vae_latent}"
                        f"_t{int(round(CFG.ic_train_frac * 100))}"
                        f"_v{int(round(CFG.ic_validation_frac * 100))}.pt")
            src = Path(CFG.out_root) / "vae_models" / vae_name
            dst_dir = Path(cfg_a.out_root) / "vae_models"
            dst_dir.mkdir(parents=True, exist_ok=True)
            if src.exists() and not (dst_dir / vae_name).exists():
                shutil.copy(src, dst_dir / vae_name)   # reuse cached VAE
            _pex = EXTRACTORS[CFG.primary_extractor]
            st_a = run_seed_calibration(seed, cfg_a, _pex, CFG.primary_extractor)
            df_a = evaluate_seed(seed, cfg_a, _pex, st_a, CFG.primary_extractor)
            df_a["setting"] = label
            abl_frames.append(df_a)

    base = RESULTS[RESULTS.seed.isin(CFG.ablation_seeds)].copy()   # reused main run
    base["setting"] = f"d={CFG.pca_dim}"
    ABL = pd.concat(abl_frames + [base], ignore_index=True)
    ABL.to_csv(Path(CFG.out_root, "aggregate", "pca_dim_sweep.csv"), index=False)

    order = [f"d={d}" for d in sorted(set(CFG.ablation_dims) | {CFG.pca_dim})]
    abl_run = (ABL[ABL.method != "VAE"]
               .groupby(["setting", "seed", "setup", "block", "method"],
                        as_index=False)
               .agg(arl1=("delay", "mean")))
    tbl = abl_run.pivot_table(index="method", columns="setting",
                              values="arl1").reindex(columns=order)
    print(f"\nUniform PCA-dim sweep: RMDT (windows) over all setups and "
          f"feature blocks, seeds {CFG.ablation_seeds} (same nominal validation AIET per d):")
    display(tbl.round(2))
    # rank stability: Spearman of the per-method ordering vs the main run
    ref = tbl[f"d={CFG.pca_dim}"]
    for c in order:
        if c != f"d={CFG.pca_dim}":
            print(f"  rank corr(d={CFG.pca_dim} vs {c}) = "
                  f"{ref.corr(tbl[c], method='spearman'):.3f}")

    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    xs = [int(c.split('=')[1]) for c in order]
    for m in tbl.index:
        ax.plot(xs, tbl.loc[m, order].values, marker="o", label=m,
                color=colors.get(m, "gray"))
    ax.set_xticks(xs)
    ax.set_xlabel("retained PCs per block (uniform across blocks)")
    ax.set_ylabel("RMDT (windows)")
    ax.set_title("Detection delay vs PCA dimension (thresholds recalibrated "
                 "per setting; all blocks share d)")
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(Path(CFG.out_root, "aggregate", "pca_dim_sweep.png"), dpi=120)
    plt.show()
else:
    print("run_pca_ablation = False -> PCA-dim sweep skipped")
